In [70]:
# ============================================================
# AIPL 2026 IPL MATCH FORECASTING
# CLEAN LEAKAGE-SAFE COMPETITION PIPELINE
# Target: strong 1.20 - 1.25 log loss range
# ============================================================

!pip install lightgbm xgboost catboost -q

import os
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [71]:
# ============================================================
# 1. UNZIP AND FIND DATA FILES
# ============================================================

ZIP_PATH = "/content/chart_aipl-2026-ipl-match-forecast-cleaned.zip"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError("Upload chart_aipl-2026-ipl-match-forecast-cleaned.zip to Colab first.")

EXTRACT_DIR = "/content/aipl_data"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

DATA_DIR = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if "train_IPL.csv" in files:
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find train_IPL.csv inside zip.")

print("DATA_DIR:", DATA_DIR)
print(os.listdir(DATA_DIR))

DATA_DIR: /content/aipl_data/train_bundle
['team_name_mapping.csv', 'README.md', 'public_lb_matches.csv', 'train_IPL.csv', 'numeric_cleaning_summary.csv', 'schedule.csv', 'sample_submission.csv', 'AIPL_2026_starter_model.ipynb', 'match_level_features.csv', 'CLEANING_REPORT.md']


In [72]:
# ============================================================
# 2. LOAD FILES
# ============================================================

train = pd.read_csv(os.path.join(DATA_DIR, "train_IPL.csv"), low_memory=False)
public_lb = pd.read_csv(os.path.join(DATA_DIR, "public_lb_matches.csv"))
schedule = pd.read_csv(os.path.join(DATA_DIR, "schedule.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

mapping_path = os.path.join(DATA_DIR, "team_name_mapping.csv")
team_map = {}

if os.path.exists(mapping_path):
    mapping = pd.read_csv(mapping_path)
    team_map = dict(zip(mapping["old_name"], mapping["clean_name"]))

def clean_team(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    return team_map.get(x, x)

team_cols_train = ["Bat First", "Bat Second", "toss_winner", "match_won_by"]
for col in team_cols_train:
    if col in train.columns:
        train[col] = train[col].apply(clean_team)

for df in [public_lb, schedule]:
    for col in ["team_a", "team_b", "toss_winner"]:
        if col in df.columns:
            df[col] = df[col].apply(clean_team)

train["Date"] = pd.to_datetime(train["Date"])
public_lb["date"] = pd.to_datetime(public_lb["date"])
schedule["date"] = pd.to_datetime(schedule["date"])

print("train:", train.shape)
print("public_lb:", public_lb.shape)
print("schedule:", schedule.shape)
print("sample:", sample.shape)

train: (272704, 38)
public_lb: (48, 9)
schedule: (5, 6)
sample: (53, 5)


In [73]:
# ============================================================
# 3. CREATE MATCH-LEVEL TABLE SAFELY FROM BALL-BY-BALL DATA
# ============================================================

def build_match_table(df):
    df = df.copy()
    df["_row_id"] = np.arange(len(df))

    const_cols = [
        "Match ID", "Date", "season", "Venue", "city",
        "Bat First", "Bat Second", "toss_winner", "toss_decision",
        "result_type", "match_won_by"
    ]

    match_const = (
        df.sort_values("_row_id")
          .groupby("Match ID")[const_cols]
          .first()
          .reset_index(drop=True)
    )

    last_ball = (
        df.sort_values("_row_id")
          .groupby(["Match ID", "Innings"])
          .tail(1)
    )

    end_stats = last_ball.pivot(
        index="Match ID",
        columns="Innings",
        values=["Innings Runs", "Innings Wickets", "Balls Remaining"]
    )

    end_stats.columns = [
        f"inn{int(inn)}_{name.lower().replace(' ', '_')}"
        for name, inn in end_stats.columns
    ]

    legal_balls = (
        df.groupby(["Match ID", "Innings"])["Valid Ball"]
          .sum()
          .unstack()
          .rename(columns={1: "inn1_legal_balls", 2: "inn2_legal_balls"})
    )

    match = match_const.merge(end_stats.reset_index(), on="Match ID", how="left")
    match = match.merge(legal_balls.reset_index(), on="Match ID", how="left")

    return match

match_table = build_match_table(train)

print(match_table.shape)
match_table.head()

(1145, 35)


,Match ID,Date,season,Venue,city,Bat First,Bat Second,toss_winner,toss_decision,result_type,...,inn3_balls_remaining,inn4_balls_remaining,inn5_balls_remaining,inn6_balls_remaining,inn1_legal_balls,inn2_legal_balls,3,4,5,6
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,normal,...,NaN,NaN,NaN,NaN,120.0,91.0,NaN,NaN,NaN,NaN
1,335983,2008-04-19,2007/08,Punjab Cricket Association Stadium,Chandigarh,Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,normal,...,NaN,NaN,NaN,NaN,120.0,120.0,NaN,NaN,NaN,NaN
2,335984,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,normal,...,NaN,NaN,NaN,NaN,120.0,91.0,NaN,NaN,NaN,NaN
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,normal,...,NaN,NaN,NaN,NaN,120.0,118.0,NaN,NaN,NaN,NaN
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,bat,normal,...,NaN,NaN,NaN,NaN,112.0,114.0,NaN,NaN,NaN,NaN


In [74]:
# ============================================================
# 4. DERIVE 4-CLASS TARGET LABEL
# ============================================================

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_ORDER)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

def derive_label(row):
    result_type = str(row["result_type"]).lower()

    if result_type in ["tie", "no result", "nan"]:
        return np.nan

    winner = row["match_won_by"]
    bat_first = row["Bat First"]
    bat_second = row["Bat Second"]

    if pd.isna(winner):
        return np.nan

    inn1_runs = row["inn1_innings_runs"]
    inn2_runs = row["inn2_innings_runs"]
    inn2_wickets = row["inn2_innings_wickets"]

    if winner == bat_first:
        margin_runs = inn1_runs - inn2_runs
        return "A_big" if margin_runs > 20 else "A_small"

    elif winner == bat_second:
        wickets_remaining = 10 - inn2_wickets
        return "B_big" if wickets_remaining >= 6 else "B_small"

    return np.nan

match_table["target_label"] = match_table.apply(derive_label, axis=1)

labelable_matches = match_table.dropna(subset=["target_label"]).copy()
labelable_matches = labelable_matches.sort_values(["Date", "Match ID"]).reset_index(drop=True)

print("Total matches:", match_table["Match ID"].nunique())
print("Labelable matches:", len(labelable_matches))
print(labelable_matches["target_label"].value_counts())

Total matches: 1145
Labelable matches: 1124
target_label
B_big      401
A_big      273
A_small    240
B_small    210
Name: count, dtype: int64


In [75]:
# ============================================================
# 5. BUILD TEAM-MATCH HISTORY TABLE
# This is used only as historical information before each match.
# ============================================================

def build_team_history(df, match_table):
    df = df.copy()
    df["_row_id"] = np.arange(len(df))

    rows = []

    valid_matches = set(match_table["Match ID"].unique())
    match_lookup = match_table.set_index("Match ID").to_dict("index")

    for match_id, g in df.groupby("Match ID"):
        if match_id not in valid_matches:
            continue

        meta = match_lookup[match_id]

        if pd.isna(meta.get("target_label", np.nan)):
            continue

        date = meta["Date"]
        venue = meta["Venue"]
        city = meta["city"]
        bat_first = meta["Bat First"]
        bat_second = meta["Bat Second"]
        winner = meta["match_won_by"]

        for inn, bat_team, bowl_team in [
            (1, bat_first, bat_second),
            (2, bat_second, bat_first)
        ]:
            inn_df = g[g["Innings"] == inn].copy()
            if len(inn_df) == 0:
                continue

            legal = inn_df["Valid Ball"].sum()
            total_runs = inn_df["Runs From Ball"].sum()
            batter_runs = inn_df["Batter Runs"].sum()
            wickets = inn_df["Wicket"].sum()

            legal_df = inn_df[inn_df["Valid Ball"] == 1]

            dots = ((legal_df["Runs From Ball"] == 0)).sum()
            fours = ((inn_df["Batter Runs"] == 4)).sum()
            sixes = ((inn_df["Batter Runs"] == 6)).sum()
            boundaries = fours + sixes

            rows.append({
                "Match ID": match_id,
                "Date": date,
                "Venue": venue,
                "city": city,
                "team": bat_team,
                "opponent": bowl_team,
                "bat_runs": total_runs,
                "bat_batter_runs": batter_runs,
                "bat_legal_balls": legal,
                "bat_wickets_lost": wickets,
                "bat_dots": dots,
                "bat_fours": fours,
                "bat_sixes": sixes,
                "bat_boundaries": boundaries,
                "bowl_runs_conceded": np.nan,
                "bowl_legal_balls": np.nan,
                "bowl_wickets_taken": np.nan,
                "won": int(winner == bat_team)
            })

            rows.append({
                "Match ID": match_id,
                "Date": date,
                "Venue": venue,
                "city": city,
                "team": bowl_team,
                "opponent": bat_team,
                "bat_runs": np.nan,
                "bat_batter_runs": np.nan,
                "bat_legal_balls": np.nan,
                "bat_wickets_lost": np.nan,
                "bat_dots": np.nan,
                "bat_fours": np.nan,
                "bat_sixes": np.nan,
                "bat_boundaries": np.nan,
                "bowl_runs_conceded": total_runs,
                "bowl_legal_balls": legal,
                "bowl_wickets_taken": wickets,
                "won": int(winner == bowl_team)
            })

    hist = pd.DataFrame(rows)

    # Combine batting and bowling rows per team-match
    agg = {
        "Date": "first",
        "Venue": "first",
        "city": "first",
        "team": "first",
        "opponent": "first",
        "bat_runs": "max",
        "bat_batter_runs": "max",
        "bat_legal_balls": "max",
        "bat_wickets_lost": "max",
        "bat_dots": "max",
        "bat_fours": "max",
        "bat_sixes": "max",
        "bat_boundaries": "max",
        "bowl_runs_conceded": "max",
        "bowl_legal_balls": "max",
        "bowl_wickets_taken": "max",
        "won": "max"
    }

    hist = (
        hist.groupby(["Match ID", "team"], as_index=False)
            .agg(agg)
            .sort_values(["Date", "Match ID"])
            .reset_index(drop=True)
    )

    return hist

team_history = build_team_history(train, labelable_matches)

print(team_history.shape)
team_history.head()

(2248, 18)


,Match ID,Date,Venue,city,team,opponent,bat_runs,bat_batter_runs,bat_legal_balls,bat_wickets_lost,bat_dots,bat_fours,bat_sixes,bat_boundaries,bowl_runs_conceded,bowl_legal_balls,bowl_wickets_taken,won
0,335982,2008-04-18,M Chinnaswamy Stadium,Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,222.0,205.0,120.0,3.0,36.0,15.0,14.0,29.0,82.0,91.0,10.0,1
1,335982,2008-04-18,M Chinnaswamy Stadium,Bengaluru,Royal Challengers Bengaluru,Kolkata Knight Riders,82.0,63.0,91.0,10.0,50.0,3.0,3.0,6.0,222.0,120.0,3.0,0
2,335983,2008-04-19,Punjab Cricket Association Stadium,Chandigarh,Chennai Super Kings,Punjab Kings,240.0,234.0,120.0,5.0,34.0,20.0,16.0,36.0,207.0,120.0,4.0,1
3,335983,2008-04-19,Punjab Cricket Association Stadium,Chandigarh,Punjab Kings,Chennai Super Kings,207.0,196.0,120.0,4.0,23.0,18.0,9.0,27.0,240.0,120.0,5.0,0
4,335984,2008-04-19,Feroz Shah Kotla,Delhi,Delhi Capitals,Rajasthan Royals,132.0,122.0,91.0,1.0,31.0,18.0,1.0,19.0,129.0,120.0,8.0,1


In [76]:
# ============================================================
# 6. FEATURE ENGINEERING HELPERS
# All features use only data BEFORE the match date.
# ============================================================

def safe_div(a, b):
    if b is None or pd.isna(b) or b == 0:
        return np.nan
    return a / b

def summarize_team(prior_team_history, team, prefix, date, n_recent=5):
    h = prior_team_history[prior_team_history["team"] == team].sort_values("Date")
    recent = h.tail(n_recent)

    out = {}

    out[f"{prefix}_matches_seen"] = len(h)
    out[f"{prefix}_recent_matches"] = len(recent)

    if len(h) > 0:
        out[f"{prefix}_days_since_last_match"] = (date - h["Date"].max()).days
    else:
        out[f"{prefix}_days_since_last_match"] = np.nan

    if len(recent) == 0:
        cols = [
            "win_rate", "bat_run_rate", "bat_sr", "boundary_rate", "dot_rate",
            "avg_wickets_lost", "bowl_economy", "bowl_wicket_rate"
        ]
        for c in cols:
            out[f"{prefix}_{c}_recent5"] = np.nan
        return out

    bat_runs = recent["bat_runs"].sum(skipna=True)
    bat_batter_runs = recent["bat_batter_runs"].sum(skipna=True)
    bat_legal = recent["bat_legal_balls"].sum(skipna=True)
    bat_boundaries = recent["bat_boundaries"].sum(skipna=True)
    bat_dots = recent["bat_dots"].sum(skipna=True)
    bat_wkts = recent["bat_wickets_lost"].mean(skipna=True)

    bowl_runs = recent["bowl_runs_conceded"].sum(skipna=True)
    bowl_legal = recent["bowl_legal_balls"].sum(skipna=True)
    bowl_wkts = recent["bowl_wickets_taken"].sum(skipna=True)

    out[f"{prefix}_win_rate_recent5"] = recent["won"].mean()
    out[f"{prefix}_bat_run_rate_recent5"] = safe_div(bat_runs, bat_legal)
    out[f"{prefix}_bat_sr_recent5"] = safe_div(bat_batter_runs * 100, bat_legal)
    out[f"{prefix}_boundary_rate_recent5"] = safe_div(bat_boundaries, bat_legal)
    out[f"{prefix}_dot_rate_recent5"] = safe_div(bat_dots, bat_legal)
    out[f"{prefix}_avg_wickets_lost_recent5"] = bat_wkts
    out[f"{prefix}_bowl_economy_recent5"] = safe_div(bowl_runs, bowl_legal / 6 if bowl_legal else np.nan)
    out[f"{prefix}_bowl_wicket_rate_recent5"] = safe_div(bowl_wkts, bowl_legal)

    return out

def summarize_venue(prior_matches, venue, city):
    exact = prior_matches[prior_matches["Venue"] == venue]
    city_df = prior_matches[prior_matches["city"] == city]

    if len(exact) >= 5:
        h = exact
    elif len(city_df) >= 5:
        h = city_df
    else:
        h = prior_matches

    out = {}

    if len(h) == 0:
        for c in [
            "matches", "avg_inn1_runs", "avg_inn2_runs", "avg_total_runs",
            "avg_inn1_wickets", "avg_inn2_wickets",
            "chase_win_rate", "bat_first_win_rate", "toss_bat_rate"
        ]:
            out[f"venue_{c}"] = np.nan
        return out

    out["venue_matches"] = len(h)
    out["venue_avg_inn1_runs"] = h["inn1_innings_runs"].mean()
    out["venue_avg_inn2_runs"] = h["inn2_innings_runs"].mean()
    out["venue_avg_total_runs"] = (
        h["inn1_innings_runs"] + h["inn2_innings_runs"]
    ).mean()

    out["venue_avg_inn1_wickets"] = h["inn1_innings_wickets"].mean()
    out["venue_avg_inn2_wickets"] = h["inn2_innings_wickets"].mean()

    out["venue_chase_win_rate"] = (h["match_won_by"] == h["Bat Second"]).mean()
    out["venue_bat_first_win_rate"] = (h["match_won_by"] == h["Bat First"]).mean()
    out["venue_toss_bat_rate"] = (h["toss_decision"] == "bat").mean()

    return out

def summarize_h2h(prior_matches, team_a, team_b):
    mask = (
        ((prior_matches["Bat First"] == team_a) & (prior_matches["Bat Second"] == team_b)) |
        ((prior_matches["Bat First"] == team_b) & (prior_matches["Bat Second"] == team_a))
    )

    h = prior_matches[mask]

    out = {}

    out["h2h_matches"] = len(h)

    if len(h) == 0:
        out["h2h_team_a_win_rate"] = np.nan
        return out

    out["h2h_team_a_win_rate"] = (h["match_won_by"] == team_a).mean()

    return out

def make_feature_row(date, team_a, team_b, venue, city, toss_winner_is_A, toss_decision_bat):
    prior_matches = labelable_matches[labelable_matches["Date"] < date].copy()
    prior_team_history = team_history[team_history["Date"] < date].copy()

    row = {}

    # identifiers as categorical features
    row["team_a"] = team_a
    row["team_b"] = team_b
    row["venue"] = venue
    row["city"] = city

    # date features
    row["year"] = date.year
    row["month"] = date.month
    row["dayofyear"] = date.dayofyear

    # toss features
    row["toss_winner_is_A"] = int(toss_winner_is_A)
    row["toss_decision_bat"] = int(toss_decision_bat)

    # team rolling features
    row.update(summarize_team(prior_team_history, team_a, "A", date, n_recent=5))
    row.update(summarize_team(prior_team_history, team_b, "B", date, n_recent=5))

    # relative differences
    for metric in [
        "win_rate_recent5",
        "bat_run_rate_recent5",
        "bat_sr_recent5",
        "boundary_rate_recent5",
        "dot_rate_recent5",
        "avg_wickets_lost_recent5",
        "bowl_economy_recent5",
        "bowl_wicket_rate_recent5"
    ]:
        a_col = f"A_{metric}"
        b_col = f"B_{metric}"
        row[f"diff_{metric}"] = row.get(a_col, np.nan) - row.get(b_col, np.nan)

    # venue and h2h
    row.update(summarize_venue(prior_matches, venue, city))
    row.update(summarize_h2h(prior_matches, team_a, team_b))

    return row

In [77]:
# ============================================================
# 7. SYMMETRICAL TRAINING DATA AUGMENTATION
# This fixes the Team A = Bat First historical asymmetry.
# ============================================================

def flip_label(label):
    side, size = label.split("_")
    new_side = "B" if side == "A" else "A"
    return f"{new_side}_{size}"

train_rows = []

for _, r in labelable_matches.iterrows():
    date = r["Date"]
    venue = r["Venue"]
    city = r["city"]

    # Perspective 1:
    # Team A = Bat First, Team B = Bat Second
    team_a = r["Bat First"]
    team_b = r["Bat Second"]

    toss_winner_is_A = int(r["toss_winner"] == team_a)
    toss_decision_bat = int(r["toss_decision"] == "bat")

    feat = make_feature_row(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=toss_winner_is_A,
        toss_decision_bat=toss_decision_bat
    )

    feat["Date"] = date
    feat["match_id"] = r["Match ID"]
    feat["target"] = r["target_label"]
    train_rows.append(feat)

    # Perspective 2:
    # Team A = Bat Second, Team B = Bat First
    team_a_flip = r["Bat Second"]
    team_b_flip = r["Bat First"]

    toss_winner_is_A_flip = int(r["toss_winner"] == team_a_flip)

    feat_flip = make_feature_row(
        date=date,
        team_a=team_a_flip,
        team_b=team_b_flip,
        venue=venue,
        city=city,
        toss_winner_is_A=toss_winner_is_A_flip,
        toss_decision_bat=toss_decision_bat
    )

    feat_flip["Date"] = date
    feat_flip["match_id"] = str(r["Match ID"]) + "_flip"
    feat_flip["target"] = flip_label(r["target_label"])
    train_rows.append(feat_flip)

train_df = pd.DataFrame(train_rows)

print(train_df.shape)
print(train_df["target"].value_counts())
train_df.head()

(2248, 53)
target
A_big      674
B_big      674
B_small    450
A_small    450
Name: count, dtype: int64


,team_a,team_b,venue,city,year,month,dayofyear,toss_winner_is_A,toss_decision_bat,A_matches_seen,...,venue_avg_inn1_wickets,venue_avg_inn2_wickets,venue_chase_win_rate,venue_bat_first_win_rate,venue_toss_bat_rate,h2h_matches,h2h_team_a_win_rate,Date,match_id,target
0,Kolkata Knight Riders,Royal Challengers Bengaluru,M Chinnaswamy Stadium,Bengaluru,2008,4,109,0,0,0,...,NaN,NaN,NaN,NaN,NaN,0,NaN,2008-04-18,335982,A_big
1,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Bengaluru,2008,4,109,1,0,0,...,NaN,NaN,NaN,NaN,NaN,0,NaN,2008-04-18,335982_flip,B_big
2,Chennai Super Kings,Punjab Kings,Punjab Cricket Association Stadium,Chandigarh,2008,4,110,1,1,0,...,3.0,10.0,0.0,1.0,0.0,0,NaN,2008-04-19,335983,A_big
3,Punjab Kings,Chennai Super Kings,Punjab Cricket Association Stadium,Chandigarh,2008,4,110,0,1,0,...,3.0,10.0,0.0,1.0,0.0,0,NaN,2008-04-19,335983_flip,B_big
4,Rajasthan Royals,Delhi Capitals,Feroz Shah Kotla,Delhi,2008,4,110,1,1,0,...,3.0,10.0,0.0,1.0,0.0,0,NaN,2008-04-19,335984,B_big


In [78]:
# ============================================================
# 8. TIME-BASED VALIDATION SPLIT
# No random K-fold leakage.
# ============================================================

train_df = train_df.sort_values("Date").reset_index(drop=True)

val_mask = train_df["Date"] >= pd.Timestamp("2025-01-01")

if val_mask.sum() < 80:
    cutoff_idx = int(len(train_df) * 0.82)
    cutoff_date = train_df["Date"].iloc[cutoff_idx]
    val_mask = train_df["Date"] >= cutoff_date

dev_df = train_df[~val_mask].copy()
val_df = train_df[val_mask].copy()

print("Dev:", dev_df.shape, dev_df["Date"].min(), dev_df["Date"].max())
print("Val:", val_df.shape, val_df["Date"].min(), val_df["Date"].max())

drop_cols = ["target", "Date", "match_id"]

X_dev = dev_df.drop(columns=drop_cols)
y_dev = dev_df["target"].map(CLASS_TO_ID).values

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["target"].map(CLASS_TO_ID).values

cat_cols = X_dev.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_dev.columns if c not in cat_cols]

print("Categorical columns:", cat_cols)
print("Numeric columns:", len(num_cols))

Dev: (2152, 53) 2008-04-18 00:00:00 2024-05-26 00:00:00
Val: (96, 53) 2025-03-22 00:00:00 2025-05-01 00:00:00
Categorical columns: ['team_a', 'team_b', 'venue', 'city']
Numeric columns: 46


In [79]:
# ============================================================
# 9. PREPROCESSING
# ============================================================

try:
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=2, sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=2, sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", ohe, cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols)
    ],
    remainder="drop"
)

X_dev_enc = preprocess.fit_transform(X_dev)
X_val_enc = preprocess.transform(X_val)

print("Encoded dev shape:", X_dev_enc.shape)
print("Encoded val shape:", X_val_enc.shape)

Encoded dev shape: (2152, 150)
Encoded val shape: (96, 150)


In [80]:
# ============================================================
# 10. TRAIN MODELS
# ============================================================

models = {}

models["lgbm"] = LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=700,
    learning_rate=0.025,
    max_depth=3,
    num_leaves=15,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.8,
    reg_lambda=3.0,
    random_state=42,
    verbose=-1
)

models["xgb"] = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=600,
    learning_rate=0.025,
    max_depth=3,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.5,
    reg_lambda=3.0,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

models["cat"] = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=800,
    depth=4,
    learning_rate=0.025,
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False
)

val_preds = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_dev_enc, y_dev)
    p = model.predict_proba(X_val_enc)

    # Make sure order is 4 classes
    p = np.asarray(p)
    val_preds[name] = p

    score = log_loss(y_val, p, labels=[0, 1, 2, 3])
    print(name, "validation log loss:", score)

Training: lgbm
lgbm validation log loss: 1.5165682187608258
Training: xgb
xgb validation log loss: 1.4479837742973374
Training: cat
cat validation log loss: 1.3743931753352994


In [81]:
# ============================================================
# 11. ENSEMBLE + PROBABILITY CALIBRATION SEARCH
# This helps reduce log loss.
# ============================================================

def normalize_probs(p):
    p = np.clip(p, 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

prior_dev = np.bincount(y_dev, minlength=4) / len(y_dev)

names = list(val_preds.keys())

best = {
    "loss": 999,
    "weights": None,
    "gamma": None,
    "alpha": None
}

weight_grid = [
    {"lgbm": 1, "xgb": 1, "cat": 1},
    {"lgbm": 2, "xgb": 1, "cat": 1},
    {"lgbm": 1, "xgb": 2, "cat": 1},
    {"lgbm": 1, "xgb": 1, "cat": 2},
    {"lgbm": 2, "xgb": 2, "cat": 1},
    {"lgbm": 2, "xgb": 1, "cat": 2},
    {"lgbm": 1, "xgb": 2, "cat": 2},
]

gammas = np.linspace(0.75, 1.25, 21)
alphas = np.linspace(0.75, 1.00, 26)

for weights in weight_grid:
    total_w = sum(weights.values())

    p = np.zeros_like(list(val_preds.values())[0])

    for name in names:
        p += weights[name] * val_preds[name]

    p = p / total_w

    for gamma in gammas:
        pg = apply_gamma(p, gamma)

        for alpha in alphas:
            pf = blend_prior(pg, prior_dev, alpha)
            loss = log_loss(y_val, pf, labels=[0, 1, 2, 3])

            if loss < best["loss"]:
                best = {
                    "loss": loss,
                    "weights": weights,
                    "gamma": gamma,
                    "alpha": alpha
                }

print("BEST VALIDATION RESULT")
print(best)

BEST VALIDATION RESULT
{'loss': 1.387599585348531, 'weights': {'lgbm': 1, 'xgb': 1, 'cat': 2}, 'gamma': np.float64(0.75), 'alpha': np.float64(0.75)}


In [82]:
# ============================================================
# 12. TRAIN FINAL MODELS ON ALL TRAINING DATA
# ============================================================

X_all = train_df.drop(columns=drop_cols)
y_all = train_df["target"].map(CLASS_TO_ID).values

cat_cols = X_all.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_all.columns if c not in cat_cols]

try:
    ohe_final = OneHotEncoder(handle_unknown="ignore", min_frequency=2, sparse_output=False)
except TypeError:
    ohe_final = OneHotEncoder(handle_unknown="ignore", min_frequency=2, sparse=False)

preprocess_final = ColumnTransformer(
    transformers=[
        ("cat", ohe_final, cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols)
    ],
    remainder="drop"
)

X_all_enc = preprocess_final.fit_transform(X_all)

final_models = {}

final_models["lgbm"] = LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=700,
    learning_rate=0.025,
    max_depth=3,
    num_leaves=15,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.8,
    reg_lambda=3.0,
    random_state=42,
    verbose=-1
)

final_models["xgb"] = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=600,
    learning_rate=0.025,
    max_depth=3,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.5,
    reg_lambda=3.0,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

final_models["cat"] = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=800,
    depth=4,
    learning_rate=0.025,
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False
)

for name, model in final_models.items():
    print("Final training:", name)
    model.fit(X_all_enc, y_all)

prior_all = np.bincount(y_all, minlength=4) / len(y_all)

print("Final prior:", dict(zip(CLASS_ORDER, prior_all)))

Final training: lgbm
Final training: xgb
Final training: cat
Final prior: {'A_small': np.float64(0.2001779359430605), 'A_big': np.float64(0.2998220640569395), 'B_small': np.float64(0.2001779359430605), 'B_big': np.float64(0.2998220640569395)}


In [83]:
# ============================================================
# 13. FINAL PREDICTION FUNCTIONS
# ============================================================

def predict_raw_feature_df(feature_df):
    X_enc = preprocess_final.transform(feature_df)

    weights = best["weights"]
    total_w = sum(weights.values())

    p = np.zeros((len(feature_df), 4))

    for name, model in final_models.items():
        p += weights[name] * model.predict_proba(X_enc)

    p = p / total_w
    return normalize_probs(p)

def postprocess_probs(p):
    p = apply_gamma(p, best["gamma"])
    p = blend_prior(p, prior_all, best["alpha"])
    return normalize_probs(p)

def predict_known_toss(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    toss_winner_is_A = int(toss_winner == team_a)
    toss_decision_bat = int(toss_decision == "bat")

    feat = make_feature_row(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=toss_winner_is_A,
        toss_decision_bat=toss_decision_bat
    )

    feature_df = pd.DataFrame([feat])
    raw = predict_raw_feature_df(feature_df)
    final = postprocess_probs(raw)

    return final[0]

def get_private_toss_bat_prior(date, venue, city):
    prior_matches = labelable_matches[labelable_matches["Date"] < date].copy()

    exact = prior_matches[prior_matches["Venue"] == venue]
    city_df = prior_matches[prior_matches["city"] == city]

    if len(exact) >= 5:
        h = exact
    elif len(city_df) >= 5:
        h = city_df
    else:
        h = prior_matches

    if len(h) == 0:
        return 0.5

    p_bat = (h["toss_decision"] == "bat").mean()

    if pd.isna(p_bat):
        return 0.5

    return float(np.clip(p_bat, 0.15, 0.85))

def predict_private_pretoss(date, team_a, team_b, venue, city):
    p_bat = get_private_toss_bat_prior(date, venue, city)

    scenarios = [
        # Team A wins toss, bats
        {"toss_winner_is_A": 1, "toss_decision_bat": 1, "weight": 0.5 * p_bat},

        # Team A wins toss, fields
        {"toss_winner_is_A": 1, "toss_decision_bat": 0, "weight": 0.5 * (1 - p_bat)},

        # Team B wins toss, bats
        {"toss_winner_is_A": 0, "toss_decision_bat": 1, "weight": 0.5 * p_bat},

        # Team B wins toss, fields
        {"toss_winner_is_A": 0, "toss_decision_bat": 0, "weight": 0.5 * (1 - p_bat)},
    ]

    feature_rows = []

    for s in scenarios:
        feat = make_feature_row(
            date=date,
            team_a=team_a,
            team_b=team_b,
            venue=venue,
            city=city,
            toss_winner_is_A=s["toss_winner_is_A"],
            toss_decision_bat=s["toss_decision_bat"]
        )
        feature_rows.append(feat)

    feature_df = pd.DataFrame(feature_rows)
    raw_scenario_probs = predict_raw_feature_df(feature_df)

    weights = np.array([s["weight"] for s in scenarios]).reshape(-1, 1)
    raw_final = (raw_scenario_probs * weights).sum(axis=0, keepdims=True)

    final = postprocess_probs(raw_final)

    return final[0]

In [84]:
# ============================================================
# 14. PREDICT ALL 53 SUBMISSION ROWS
# ============================================================

pred_dict = {}

# Public LB rows: toss is known in public_lb_matches.csv
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict[match_id] = p

# Private 2026 scoring rows: toss unknown, use 4-scenario marginalization
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict[match_id] = p

print("Predictions created:", len(pred_dict))

Predictions created: 53


# V2 CELL 1 — Build stronger team history

In [85]:
# ============================================================
# VERSION 2: STRONGER TEAM HISTORY WITH MARGIN + PHASE FEATURES
# ============================================================

def safe_div(a, b):
    if b is None or pd.isna(b) or b == 0:
        return np.nan
    return a / b

def phase_stats(inn_df):
    out = {}

    phases = {
        "pp": inn_df[inn_df["Over"] <= 6],
        "mid": inn_df[(inn_df["Over"] >= 7) & (inn_df["Over"] <= 15)],
        "death": inn_df[inn_df["Over"] >= 16],
    }

    for name, d in phases.items():
        legal = d["Valid Ball"].sum()
        runs = d["Runs From Ball"].sum()
        batter_runs = d["Batter Runs"].sum()
        wkts = d["Wicket"].sum()

        legal_d = d[d["Valid Ball"] == 1]
        dots = (legal_d["Runs From Ball"] == 0).sum()
        boundaries = ((d["Batter Runs"] == 4) | (d["Batter Runs"] == 6)).sum()

        out[f"{name}_runs"] = runs
        out[f"{name}_legal_balls"] = legal
        out[f"{name}_wickets"] = wkts
        out[f"{name}_dot_rate"] = safe_div(dots, legal)
        out[f"{name}_boundary_rate"] = safe_div(boundaries, legal)
        out[f"{name}_run_rate"] = safe_div(runs, legal)

    return out


def build_team_history_v2(ball_df, match_df):
    ball_df = ball_df.copy()
    ball_df["_row_id"] = np.arange(len(ball_df))

    match_lookup = match_df.set_index("Match ID").to_dict("index")
    rows = []

    for match_id, g in ball_df.groupby("Match ID"):
        if match_id not in match_lookup:
            continue

        meta = match_lookup[match_id]

        if pd.isna(meta.get("target_label", np.nan)):
            continue

        date = meta["Date"]
        venue = meta["Venue"]
        city = meta["city"]
        bat_first = meta["Bat First"]
        bat_second = meta["Bat Second"]
        winner = meta["match_won_by"]
        label = meta["target_label"]

        is_big_margin = label.endswith("big")
        is_small_margin = label.endswith("small")

        for inn, batting_team, bowling_team in [
            (1, bat_first, bat_second),
            (2, bat_second, bat_first)
        ]:
            inn_df = g[g["Innings"] == inn].copy()

            if len(inn_df) == 0:
                continue

            legal = inn_df["Valid Ball"].sum()
            runs = inn_df["Runs From Ball"].sum()
            batter_runs = inn_df["Batter Runs"].sum()
            wickets = inn_df["Wicket"].sum()

            legal_df = inn_df[inn_df["Valid Ball"] == 1]
            dots = (legal_df["Runs From Ball"] == 0).sum()
            fours = (inn_df["Batter Runs"] == 4).sum()
            sixes = (inn_df["Batter Runs"] == 6).sum()
            boundaries = fours + sixes

            batting_won = int(winner == batting_team)
            bowling_won = int(winner == bowling_team)

            phase = phase_stats(inn_df)

            # Batting team row
            bat_row = {
                "Match ID": match_id,
                "Date": date,
                "Venue": venue,
                "city": city,
                "team": batting_team,
                "opponent": bowling_team,

                "bat_runs": runs,
                "bat_batter_runs": batter_runs,
                "bat_legal_balls": legal,
                "bat_wickets_lost": wickets,
                "bat_dots": dots,
                "bat_fours": fours,
                "bat_sixes": sixes,
                "bat_boundaries": boundaries,

                "bowl_runs_conceded": np.nan,
                "bowl_legal_balls": np.nan,
                "bowl_wickets_taken": np.nan,

                "won": batting_won,
                "big_win": int(batting_won and is_big_margin),
                "small_win": int(batting_won and is_small_margin),
                "big_loss": int((not batting_won) and is_big_margin),
                "small_loss": int((not batting_won) and is_small_margin),
            }

            for k, v in phase.items():
                bat_row[f"bat_{k}"] = v

            rows.append(bat_row)

            # Bowling team row
            bowl_row = {
                "Match ID": match_id,
                "Date": date,
                "Venue": venue,
                "city": city,
                "team": bowling_team,
                "opponent": batting_team,

                "bat_runs": np.nan,
                "bat_batter_runs": np.nan,
                "bat_legal_balls": np.nan,
                "bat_wickets_lost": np.nan,
                "bat_dots": np.nan,
                "bat_fours": np.nan,
                "bat_sixes": np.nan,
                "bat_boundaries": np.nan,

                "bowl_runs_conceded": runs,
                "bowl_legal_balls": legal,
                "bowl_wickets_taken": wickets,

                "won": bowling_won,
                "big_win": int(bowling_won and is_big_margin),
                "small_win": int(bowling_won and is_small_margin),
                "big_loss": int((not bowling_won) and is_big_margin),
                "small_loss": int((not bowling_won) and is_small_margin),
            }

            for k, v in phase.items():
                bowl_row[f"bowl_{k}"] = v

            rows.append(bowl_row)

    hist = pd.DataFrame(rows)

    agg = {}
    for c in hist.columns:
        if c in ["Match ID", "team"]:
            continue
        elif c in ["Date", "Venue", "city", "opponent"]:
            agg[c] = "first"
        else:
            agg[c] = "max"

    hist = (
        hist.groupby(["Match ID", "team"], as_index=False)
            .agg(agg)
            .sort_values(["Date", "Match ID"])
            .reset_index(drop=True)
    )

    return hist


team_history_v2 = build_team_history_v2(train, labelable_matches)

print(team_history_v2.shape)
team_history_v2.head()

(2248, 58)


,Match ID,team,Date,Venue,city,opponent,bat_runs,bat_batter_runs,bat_legal_balls,bat_wickets_lost,...,bowl_mid_wickets,bowl_mid_dot_rate,bowl_mid_boundary_rate,bowl_mid_run_rate,bowl_death_runs,bowl_death_legal_balls,bowl_death_wickets,bowl_death_dot_rate,bowl_death_boundary_rate,bowl_death_run_rate
0,335982,Kolkata Knight Riders,2008-04-18,M Chinnaswamy Stadium,Bengaluru,Royal Challengers Bengaluru,222.0,205.0,120.0,3.0,...,5.0,0.518519,0.092593,1.018519,1.0,1.0,1.0,1.000000,0.000000,1.000000
1,335982,Royal Challengers Bengaluru,2008-04-18,M Chinnaswamy Stadium,Bengaluru,Kolkata Knight Riders,82.0,63.0,91.0,10.0,...,1.0,0.222222,0.185185,1.722222,68.0,30.0,1.0,0.233333,0.300000,2.266667
2,335983,Chennai Super Kings,2008-04-19,Punjab Cricket Association Stadium,Chandigarh,Punjab Kings,240.0,234.0,120.0,5.0,...,2.0,0.129630,0.259259,1.888889,42.0,30.0,1.0,0.133333,0.100000,1.400000
3,335983,Punjab Kings,2008-04-19,Punjab Cricket Association Stadium,Chandigarh,Chennai Super Kings,207.0,196.0,120.0,4.0,...,3.0,0.203704,0.259259,2.000000,79.0,30.0,1.0,0.133333,0.400000,2.633333
4,335984,Delhi Capitals,2008-04-19,Feroz Shah Kotla,Delhi,Rajasthan Royals,132.0,122.0,91.0,1.0,...,5.0,0.370370,0.092593,1.037037,33.0,30.0,1.0,0.466667,0.166667,1.100000


In [86]:
# ============================================================
# VERSION 2: STRONGER FEATURE BUILDER
# ============================================================

def summarize_team_window(h, prefix, window):
    recent = h.tail(window)
    out = {}

    out[f"{prefix}_matches_seen_w{window}"] = len(h)
    out[f"{prefix}_recent_count_w{window}"] = len(recent)

    if len(recent) == 0:
        return out

    bat_runs = recent["bat_runs"].sum(skipna=True)
    bat_batter_runs = recent["bat_batter_runs"].sum(skipna=True)
    bat_legal = recent["bat_legal_balls"].sum(skipna=True)
    bat_boundaries = recent["bat_boundaries"].sum(skipna=True)
    bat_dots = recent["bat_dots"].sum(skipna=True)

    bowl_runs = recent["bowl_runs_conceded"].sum(skipna=True)
    bowl_legal = recent["bowl_legal_balls"].sum(skipna=True)
    bowl_wickets = recent["bowl_wickets_taken"].sum(skipna=True)

    out[f"{prefix}_win_rate_w{window}"] = recent["won"].mean()
    out[f"{prefix}_big_win_rate_w{window}"] = recent["big_win"].mean()
    out[f"{prefix}_small_win_rate_w{window}"] = recent["small_win"].mean()
    out[f"{prefix}_big_loss_rate_w{window}"] = recent["big_loss"].mean()
    out[f"{prefix}_small_loss_rate_w{window}"] = recent["small_loss"].mean()

    out[f"{prefix}_bat_run_rate_w{window}"] = safe_div(bat_runs, bat_legal)
    out[f"{prefix}_bat_sr_w{window}"] = safe_div(bat_batter_runs * 100, bat_legal)
    out[f"{prefix}_bat_boundary_rate_w{window}"] = safe_div(bat_boundaries, bat_legal)
    out[f"{prefix}_bat_dot_rate_w{window}"] = safe_div(bat_dots, bat_legal)
    out[f"{prefix}_avg_wickets_lost_w{window}"] = recent["bat_wickets_lost"].mean(skipna=True)

    out[f"{prefix}_bowl_economy_w{window}"] = safe_div(bowl_runs, bowl_legal / 6 if bowl_legal else np.nan)
    out[f"{prefix}_bowl_wicket_rate_w{window}"] = safe_div(bowl_wickets, bowl_legal)
    out[f"{prefix}_bowl_runs_per_ball_w{window}"] = safe_div(bowl_runs, bowl_legal)

    phase_cols = [
        "pp_run_rate", "pp_dot_rate", "pp_boundary_rate", "pp_wickets",
        "mid_run_rate", "mid_dot_rate", "mid_boundary_rate", "mid_wickets",
        "death_run_rate", "death_dot_rate", "death_boundary_rate", "death_wickets"
    ]

    for pc in phase_cols:
        bat_col = f"bat_{pc}"
        bowl_col = f"bowl_{pc}"

        if bat_col in recent.columns:
            out[f"{prefix}_bat_{pc}_w{window}"] = recent[bat_col].mean(skipna=True)

        if bowl_col in recent.columns:
            out[f"{prefix}_bowl_{pc}_w{window}"] = recent[bowl_col].mean(skipna=True)

    return out


def summarize_team_v2(prior_team_history, team, prefix, date):
    h = prior_team_history[prior_team_history["team"] == team].sort_values("Date")

    out = {}

    out[f"{prefix}_total_matches_seen"] = len(h)

    if len(h) > 0:
        out[f"{prefix}_days_since_last_match"] = (date - h["Date"].max()).days
    else:
        out[f"{prefix}_days_since_last_match"] = np.nan

    for window in [5, 10, 20]:
        out.update(summarize_team_window(h, prefix, window))

    return out


def summarize_venue_v2(prior_matches, venue, city):
    exact = prior_matches[prior_matches["Venue"] == venue]
    city_df = prior_matches[prior_matches["city"] == city]

    if len(exact) >= 5:
        h = exact
    elif len(city_df) >= 5:
        h = city_df
    else:
        h = prior_matches

    out = {}

    out["venue_history_count"] = len(h)

    if len(h) == 0:
        return out

    out["venue_avg_inn1_runs"] = h["inn1_innings_runs"].mean()
    out["venue_avg_inn2_runs"] = h["inn2_innings_runs"].mean()
    out["venue_avg_total_runs"] = (h["inn1_innings_runs"] + h["inn2_innings_runs"]).mean()

    out["venue_std_inn1_runs"] = h["inn1_innings_runs"].std()
    out["venue_std_total_runs"] = (h["inn1_innings_runs"] + h["inn2_innings_runs"]).std()

    out["venue_avg_inn1_wickets"] = h["inn1_innings_wickets"].mean()
    out["venue_avg_inn2_wickets"] = h["inn2_innings_wickets"].mean()

    out["venue_chase_win_rate"] = (h["match_won_by"] == h["Bat Second"]).mean()
    out["venue_bat_first_win_rate"] = (h["match_won_by"] == h["Bat First"]).mean()
    out["venue_toss_bat_rate"] = (h["toss_decision"] == "bat").mean()

    out["venue_big_rate"] = h["target_label"].astype(str).str.endswith("big").mean()
    out["venue_small_rate"] = h["target_label"].astype(str).str.endswith("small").mean()

    out["venue_A_big_rate"] = (h["target_label"] == "A_big").mean()
    out["venue_A_small_rate"] = (h["target_label"] == "A_small").mean()
    out["venue_B_big_rate"] = (h["target_label"] == "B_big").mean()
    out["venue_B_small_rate"] = (h["target_label"] == "B_small").mean()

    return out


def summarize_h2h_v2(prior_matches, team_a, team_b):
    mask = (
        ((prior_matches["Bat First"] == team_a) & (prior_matches["Bat Second"] == team_b)) |
        ((prior_matches["Bat First"] == team_b) & (prior_matches["Bat Second"] == team_a))
    )

    h = prior_matches[mask].sort_values("Date").tail(10)

    out = {}
    out["h2h_count"] = len(h)

    if len(h) == 0:
        return out

    out["h2h_A_win_rate"] = (h["match_won_by"] == team_a).mean()
    out["h2h_big_rate"] = h["target_label"].astype(str).str.endswith("big").mean()
    out["h2h_small_rate"] = h["target_label"].astype(str).str.endswith("small").mean()

    return out


def make_feature_row_v2(date, team_a, team_b, venue, city, toss_winner_is_A, toss_decision_bat):
    prior_matches = labelable_matches[labelable_matches["Date"] < date].copy()
    prior_team_history = team_history_v2[team_history_v2["Date"] < date].copy()

    row = {}

    row["team_a"] = str(team_a)
    row["team_b"] = str(team_b)
    row["venue"] = str(venue)
    row["city"] = str(city)

    row["year"] = date.year
    row["month"] = date.month
    row["dayofyear"] = date.dayofyear

    row["toss_winner_is_A"] = int(toss_winner_is_A)
    row["toss_decision_bat"] = int(toss_decision_bat)

    row.update(summarize_team_v2(prior_team_history, team_a, "A", date))
    row.update(summarize_team_v2(prior_team_history, team_b, "B", date))

    # Difference features
    for c in list(row.keys()):
        if c.startswith("A_"):
            b_col = "B_" + c[2:]
            if b_col in row:
                row["diff_" + c[2:]] = row[c] - row[b_col]

    row.update(summarize_venue_v2(prior_matches, venue, city))
    row.update(summarize_h2h_v2(prior_matches, team_a, team_b))

    return row

In [87]:
# ============================================================
# VERSION 2: BUILD AUGMENTED TRAINING DATA
# ============================================================

def flip_label(label):
    side, size = label.split("_")
    new_side = "B" if side == "A" else "A"
    return f"{new_side}_{size}"


train_rows_v2 = []

for _, r in labelable_matches.iterrows():
    date = r["Date"]
    venue = r["Venue"]
    city = r["city"]

    # Original perspective
    team_a = r["Bat First"]
    team_b = r["Bat Second"]

    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(r["toss_winner"] == team_a),
        toss_decision_bat=int(r["toss_decision"] == "bat")
    )

    feat["Date"] = date
    feat["match_id"] = r["Match ID"]
    feat["target"] = r["target_label"]
    train_rows_v2.append(feat)

    # Flipped perspective
    team_a_flip = r["Bat Second"]
    team_b_flip = r["Bat First"]

    feat_flip = make_feature_row_v2(
        date=date,
        team_a=team_a_flip,
        team_b=team_b_flip,
        venue=venue,
        city=city,
        toss_winner_is_A=int(r["toss_winner"] == team_a_flip),
        toss_decision_bat=int(r["toss_decision"] == "bat")
    )

    feat_flip["Date"] = date
    feat_flip["match_id"] = str(r["Match ID"]) + "_flip"
    feat_flip["target"] = flip_label(r["target_label"])
    train_rows_v2.append(feat_flip)

train_df_v2 = pd.DataFrame(train_rows_v2)
train_df_v2 = train_df_v2.sort_values("Date").reset_index(drop=True)

print(train_df_v2.shape)
print(train_df_v2["target"].value_counts())
train_df_v2.head()

(2248, 390)
target
A_big      674
B_big      674
B_small    450
A_small    450
Name: count, dtype: int64


,team_a,team_b,venue,city,year,month,dayofyear,toss_winner_is_A,toss_decision_bat,A_total_matches_seen,...,diff_bowl_death_run_rate_w20,diff_bat_death_dot_rate_w20,diff_bowl_death_dot_rate_w20,diff_bat_death_boundary_rate_w20,diff_bowl_death_boundary_rate_w20,diff_bat_death_wickets_w20,diff_bowl_death_wickets_w20,h2h_A_win_rate,h2h_big_rate,h2h_small_rate
0,Kolkata Knight Riders,Royal Challengers Bengaluru,M Chinnaswamy Stadium,Bengaluru,2008,4,109,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Bengaluru,2008,4,109,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Chennai Super Kings,Punjab Kings,Punjab Cricket Association Stadium,Chandigarh,2008,4,110,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Punjab Kings,Chennai Super Kings,Punjab Cricket Association Stadium,Chandigarh,2008,4,110,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Rajasthan Royals,Delhi Capitals,Feroz Shah Kotla,Delhi,2008,4,110,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
# ============================================================
# VERSION 2: TIME VALIDATION
# ============================================================

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_ORDER)}

val_mask = train_df_v2["Date"] >= pd.Timestamp("2025-01-01")

if val_mask.sum() < 80:
    cutoff_idx = int(len(train_df_v2) * 0.82)
    cutoff_date = train_df_v2["Date"].iloc[cutoff_idx]
    val_mask = train_df_v2["Date"] >= cutoff_date

dev_df_v2 = train_df_v2[~val_mask].copy()
val_df_v2 = train_df_v2[val_mask].copy()

drop_cols = ["target", "Date", "match_id"]

X_dev_v2 = dev_df_v2.drop(columns=drop_cols)
y_dev_v2 = dev_df_v2["target"].map(CLASS_TO_ID).values

X_val_v2 = val_df_v2.drop(columns=drop_cols)
y_val_v2 = val_df_v2["target"].map(CLASS_TO_ID).values

cat_cols_v2 = X_dev_v2.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v2:
    X_dev_v2[c] = X_dev_v2[c].fillna("Unknown").astype(str)
    X_val_v2[c] = X_val_v2[c].fillna("Unknown").astype(str)

cat_features_v2 = [X_dev_v2.columns.get_loc(c) for c in cat_cols_v2]

print("Dev:", dev_df_v2.shape, dev_df_v2["Date"].min(), dev_df_v2["Date"].max())
print("Val:", val_df_v2.shape, val_df_v2["Date"].min(), val_df_v2["Date"].max())
print("Features:", X_dev_v2.shape[1])
print("Categorical:", cat_cols_v2)

Dev: (2152, 390) 2008-04-18 00:00:00 2024-05-26 00:00:00
Val: (96, 390) 2025-03-22 00:00:00 2025-05-01 00:00:00
Features: 387
Categorical: ['team_a', 'team_b', 'venue', 'city']


In [89]:
# ============================================================
# VERSION 2: CATBOOST ONLY + PROBABILITY TUNING
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss

cat_v2 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=1200,
    depth=4,
    learning_rate=0.018,
    l2_leaf_reg=8,
    random_strength=1.5,
    bagging_temperature=0.7,
    border_count=128,
    random_seed=42,
    verbose=100
)

cat_v2.fit(
    X_dev_v2,
    y_dev_v2,
    cat_features=cat_features_v2,
    eval_set=(X_val_v2, y_val_v2),
    use_best_model=True
)

val_raw_v2 = cat_v2.predict_proba(X_val_v2)

base_loss = log_loss(y_val_v2, val_raw_v2, labels=[0, 1, 2, 3])
print("Raw CatBoost V2 validation log loss:", base_loss)


def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

prior_dev_v2 = np.bincount(y_dev_v2, minlength=4) / len(y_dev_v2)

best_v2 = {
    "loss": 999,
    "gamma": None,
    "alpha": None
}

for gamma in np.linspace(0.60, 1.30, 36):
    pg = apply_gamma(val_raw_v2, gamma)

    for alpha in np.linspace(0.55, 1.00, 46):
        pf = blend_prior(pg, prior_dev_v2, alpha)
        loss = log_loss(y_val_v2, pf, labels=[0, 1, 2, 3])

        if loss < best_v2["loss"]:
            best_v2 = {
                "loss": loss,
                "gamma": gamma,
                "alpha": alpha
            }

print("BEST V2 VALIDATION RESULT:")
print(best_v2)

0:	learn: 1.3855518	test: 1.3860294	best: 1.3860294 (0)	total: 106ms	remaining: 2m 6s
100:	learn: 1.3457417	test: 1.3675554	best: 1.3674897 (92)	total: 4.86s	remaining: 52.8s
200:	learn: 1.3225361	test: 1.3680148	best: 1.3667480 (120)	total: 9.24s	remaining: 45.9s
300:	learn: 1.3025053	test: 1.3679984	best: 1.3667480 (120)	total: 14.3s	remaining: 42.6s
400:	learn: 1.2815164	test: 1.3672489	best: 1.3667480 (120)	total: 18.9s	remaining: 37.7s
500:	learn: 1.2566639	test: 1.3693758	best: 1.3667480 (120)	total: 23.1s	remaining: 32.2s
600:	learn: 1.2287115	test: 1.3691305	best: 1.3667480 (120)	total: 28.1s	remaining: 28s
700:	learn: 1.1956541	test: 1.3700189	best: 1.3667480 (120)	total: 32.4s	remaining: 23s
800:	learn: 1.1654225	test: 1.3726727	best: 1.3667480 (120)	total: 37.2s	remaining: 18.5s
900:	learn: 1.1386531	test: 1.3755114	best: 1.3667480 (120)	total: 41.5s	remaining: 13.8s
1000:	learn: 1.1133788	test: 1.3791728	best: 1.3667480 (120)	total: 45.6s	remaining: 9.06s
1100:	learn: 1.087

In [90]:
# ============================================================
# VERSION 2: FINAL TRAINING ON ALL DATA
# ============================================================

X_all_v2 = train_df_v2.drop(columns=drop_cols)
y_all_v2 = train_df_v2["target"].map(CLASS_TO_ID).values

cat_cols_v2_final = X_all_v2.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v2_final:
    X_all_v2[c] = X_all_v2[c].fillna("Unknown").astype(str)

cat_features_v2_final = [X_all_v2.columns.get_loc(c) for c in cat_cols_v2_final]

final_cat_v2 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=cat_v2.best_iteration_ if cat_v2.best_iteration_ is not None else 900,
    depth=4,
    learning_rate=0.018,
    l2_leaf_reg=8,
    random_strength=1.5,
    bagging_temperature=0.7,
    border_count=128,
    random_seed=42,
    verbose=100
)

final_cat_v2.fit(
    X_all_v2,
    y_all_v2,
    cat_features=cat_features_v2_final
)

prior_all_v2 = np.bincount(y_all_v2, minlength=4) / len(y_all_v2)

print("Final V2 prior:")
print(dict(zip(CLASS_ORDER, prior_all_v2)))

0:	learn: 1.3854883	total: 44.4ms	remaining: 5.28s
100:	learn: 1.3453909	total: 3.87s	remaining: 729ms
119:	learn: 1.3417702	total: 5.08s	remaining: 0us
Final V2 prior:
{'A_small': np.float64(0.2001779359430605), 'A_big': np.float64(0.2998220640569395), 'B_small': np.float64(0.2001779359430605), 'B_big': np.float64(0.2998220640569395)}


In [91]:
# ============================================================
# VERSION 2: PREDICTION FUNCTIONS
# ============================================================

def predict_raw_feature_df_v2(feature_df):
    feature_df = feature_df.copy()

    for c in cat_cols_v2_final:
        if c not in feature_df.columns:
            feature_df[c] = "Unknown"
        feature_df[c] = feature_df[c].fillna("Unknown").astype(str)

    # Align columns exactly
    for c in X_all_v2.columns:
        if c not in feature_df.columns:
            feature_df[c] = np.nan

    feature_df = feature_df[X_all_v2.columns]

    p = final_cat_v2.predict_proba(feature_df)
    return normalize_probs(p)

def postprocess_probs_v2(p):
    p = apply_gamma(p, best_v2["gamma"])
    p = blend_prior(p, prior_all_v2, best_v2["alpha"])
    return normalize_probs(p)

def predict_known_toss_v2(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(toss_winner == team_a),
        toss_decision_bat=int(toss_decision == "bat")
    )

    feature_df = pd.DataFrame([feat])
    raw = predict_raw_feature_df_v2(feature_df)
    final = postprocess_probs_v2(raw)

    return final[0]

def get_private_toss_bat_prior_v2(date, venue, city):
    prior_matches = labelable_matches[labelable_matches["Date"] < date].copy()

    exact = prior_matches[prior_matches["Venue"] == venue]
    city_df = prior_matches[prior_matches["city"] == city]

    if len(exact) >= 5:
        h = exact
    elif len(city_df) >= 5:
        h = city_df
    else:
        h = prior_matches

    if len(h) == 0:
        return 0.5

    p_bat = (h["toss_decision"] == "bat").mean()

    if pd.isna(p_bat):
        return 0.5

    return float(np.clip(p_bat, 0.15, 0.85))

def predict_private_pretoss_v2(date, team_a, team_b, venue, city):
    p_bat = get_private_toss_bat_prior_v2(date, venue, city)

    scenarios = [
        {"toss_winner_is_A": 1, "toss_decision_bat": 1, "weight": 0.5 * p_bat},
        {"toss_winner_is_A": 1, "toss_decision_bat": 0, "weight": 0.5 * (1 - p_bat)},
        {"toss_winner_is_A": 0, "toss_decision_bat": 1, "weight": 0.5 * p_bat},
        {"toss_winner_is_A": 0, "toss_decision_bat": 0, "weight": 0.5 * (1 - p_bat)},
    ]

    feature_rows = []

    for s in scenarios:
        feat = make_feature_row_v2(
            date=date,
            team_a=team_a,
            team_b=team_b,
            venue=venue,
            city=city,
            toss_winner_is_A=s["toss_winner_is_A"],
            toss_decision_bat=s["toss_decision_bat"]
        )
        feature_rows.append(feat)

    feature_df = pd.DataFrame(feature_rows)
    raw_scenario_probs = predict_raw_feature_df_v2(feature_df)

    weights = np.array([s["weight"] for s in scenarios]).reshape(-1, 1)
    raw_final = (raw_scenario_probs * weights).sum(axis=0, keepdims=True)

    final = postprocess_probs_v2(raw_final)

    return final[0]

In [92]:
# ============================================================
# VERSION 2: FINAL TRAINING ON ALL DATA
# ============================================================

X_all_v2 = train_df_v2.drop(columns=drop_cols)
y_all_v2 = train_df_v2["target"].map(CLASS_TO_ID).values

cat_cols_v2_final = X_all_v2.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v2_final:
    X_all_v2[c] = X_all_v2[c].fillna("Unknown").astype(str)

cat_features_v2_final = [X_all_v2.columns.get_loc(c) for c in cat_cols_v2_final]

final_cat_v2 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=cat_v2.best_iteration_ + 1 if cat_v2.best_iteration_ is not None else 121,
    depth=4,
    learning_rate=0.018,
    l2_leaf_reg=8,
    random_strength=1.5,
    bagging_temperature=0.7,
    border_count=128,
    random_seed=42,
    verbose=100
)

final_cat_v2.fit(
    X_all_v2,
    y_all_v2,
    cat_features=cat_features_v2_final
)

prior_all_v2 = np.bincount(y_all_v2, minlength=4) / len(y_all_v2)

print("Final V2 prior:")
print(dict(zip(CLASS_ORDER, prior_all_v2)))

0:	learn: 1.3854883	total: 54.4ms	remaining: 6.53s
100:	learn: 1.3453909	total: 3.93s	remaining: 779ms
120:	learn: 1.3416584	total: 4.68s	remaining: 0us
Final V2 prior:
{'A_small': np.float64(0.2001779359430605), 'A_big': np.float64(0.2998220640569395), 'B_small': np.float64(0.2001779359430605), 'B_big': np.float64(0.2998220640569395)}


In [93]:
# ============================================================
# CREATE submission_v2.csv
# ============================================================

import numpy as np
import pandas as pd

pred_dict_v2 = {}

# Public rows: known toss
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict_v2[match_id] = p

# Private rows: pre-toss
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict_v2[match_id] = p

submission_v2 = sample.copy()
submission_v2["match_id"] = submission_v2["match_id"].astype(str)

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]

for i, match_id in enumerate(submission_v2["match_id"]):
    if match_id not in pred_dict_v2:
        print("Missing:", match_id, "using prior fallback")
        p = prior_all_v2.copy()
    else:
        p = pred_dict_v2[match_id]

    p = normalize_probs(np.asarray(p).reshape(1, -1))[0]

    submission_v2.loc[i, "A_small"] = p[0]
    submission_v2.loc[i, "A_big"] = p[1]
    submission_v2.loc[i, "B_small"] = p[2]
    submission_v2.loc[i, "B_big"] = p[3]

assert submission_v2.shape == (53, 5)
assert submission_v2["match_id"].astype(str).tolist() == sample["match_id"].astype(str).tolist()
assert np.all(submission_v2[CLASS_ORDER].values >= 0)
assert np.all(submission_v2[CLASS_ORDER].values <= 1)
assert np.allclose(submission_v2[CLASS_ORDER].sum(axis=1), 1.0, atol=1e-6)

submission_v2.to_csv("submission_v2.csv", index=False)

print("✅ submission_v2.csv saved")
print(submission_v2.shape)
print(submission_v2[CLASS_ORDER].mean())
display(submission_v2.head())
display(submission_v2.tail())

✅ submission_v2.csv saved
(53, 5)
A_small    0.198941
A_big      0.302500
B_small    0.197412
B_big      0.301147
dtype: float64


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.202455,0.306500,0.191299,0.299746
1,1473489,0.210412,0.307598,0.191767,0.290223
2,1473490,0.201431,0.298650,0.194282,0.305637
3,1473491,0.202602,0.305004,0.191704,0.300690
4,1473492,0.200175,0.304428,0.193001,0.302397


,match_id,A_small,A_big,B_small,B_big
48,M_2026_T01,0.195267,0.293354,0.203494,0.307885
49,M_2026_T02,0.198750,0.297648,0.198525,0.305077
50,M_2026_T03,0.196740,0.300049,0.200914,0.302297
51,M_2026_T04,0.204248,0.308299,0.193001,0.294452
52,M_2026_T05,0.202115,0.301531,0.197217,0.299138


# V3 CELL 1 — Build original-only known-toss training data

In [94]:
# ============================================================
# VERSION 3: ORIGINAL-ONLY MODEL FOR PUBLIC KNOWN-TOSS MATCHES
# This is sharper for public_lb_matches because Team A = Bat First.
# ============================================================

train_rows_v3_public = []

for _, r in labelable_matches.iterrows():
    date = r["Date"]
    venue = r["Venue"]
    city = r["city"]

    team_a = r["Bat First"]
    team_b = r["Bat Second"]

    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(r["toss_winner"] == team_a),
        toss_decision_bat=int(r["toss_decision"] == "bat")
    )

    feat["Date"] = date
    feat["match_id"] = r["Match ID"]
    feat["target"] = r["target_label"]

    train_rows_v3_public.append(feat)

train_df_v3_public = pd.DataFrame(train_rows_v3_public)
train_df_v3_public = train_df_v3_public.sort_values("Date").reset_index(drop=True)

print("V3 public train shape:", train_df_v3_public.shape)
print(train_df_v3_public["target"].value_counts(normalize=True))
print(train_df_v3_public["target"].value_counts())

V3 public train shape: (1124, 390)
target
B_big      0.356762
A_big      0.242883
A_small    0.213523
B_small    0.186833
Name: proportion, dtype: float64
target
B_big      401
A_big      273
A_small    240
B_small    210
Name: count, dtype: int64


In [95]:
# ============================================================
# VERSION 3: TIME VALIDATION ORIGINAL-ONLY
# ============================================================

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_ORDER)}

drop_cols = ["target", "Date", "match_id"]

val_mask_v3 = train_df_v3_public["Date"] >= pd.Timestamp("2025-01-01")

if val_mask_v3.sum() < 30:
    cutoff_idx = int(len(train_df_v3_public) * 0.85)
    cutoff_date = train_df_v3_public["Date"].iloc[cutoff_idx]
    val_mask_v3 = train_df_v3_public["Date"] >= cutoff_date

dev_df_v3 = train_df_v3_public[~val_mask_v3].copy()
val_df_v3 = train_df_v3_public[val_mask_v3].copy()

X_dev_v3 = dev_df_v3.drop(columns=drop_cols)
y_dev_v3 = dev_df_v3["target"].map(CLASS_TO_ID).values

X_val_v3 = val_df_v3.drop(columns=drop_cols)
y_val_v3 = val_df_v3["target"].map(CLASS_TO_ID).values

cat_cols_v3 = X_dev_v3.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v3:
    X_dev_v3[c] = X_dev_v3[c].fillna("Unknown").astype(str)
    X_val_v3[c] = X_val_v3[c].fillna("Unknown").astype(str)

cat_features_v3 = [X_dev_v3.columns.get_loc(c) for c in cat_cols_v3]

print("V3 Dev:", dev_df_v3.shape, dev_df_v3["Date"].min(), dev_df_v3["Date"].max())
print("V3 Val:", val_df_v3.shape, val_df_v3["Date"].min(), val_df_v3["Date"].max())
print("V3 Features:", X_dev_v3.shape[1])
print("V3 Categorical:", cat_cols_v3)
print("V3 Val target distribution:")
print(val_df_v3["target"].value_counts())

V3 Dev: (1076, 390) 2008-04-18 00:00:00 2024-05-26 00:00:00
V3 Val: (48, 390) 2025-03-22 00:00:00 2025-05-01 00:00:00
V3 Features: 387
V3 Categorical: ['team_a', 'team_b', 'venue', 'city']
V3 Val target distribution:
target
B_big      19
A_small    11
A_big      10
B_small     8
Name: count, dtype: int64


In [ ]:
# ============================================================
# VERSION 3: CATBOOST PARAMETER SEARCH
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
import numpy as np

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

prior_dev_v3 = np.bincount(y_dev_v3, minlength=4) / len(y_dev_v3)

cat_configs_v3 = [
    {
        "name": "safe_depth3",
        "depth": 3,
        "learning_rate": 0.025,
        "l2_leaf_reg": 10,
        "random_strength": 2.0,
        "bagging_temperature": 0.8,
        "iterations": 900
    },
    {
        "name": "balanced_depth4",
        "depth": 4,
        "learning_rate": 0.018,
        "l2_leaf_reg": 8,
        "random_strength": 1.5,
        "bagging_temperature": 0.7,
        "iterations": 1100
    },
    {
        "name": "strong_reg_depth4",
        "depth": 4,
        "learning_rate": 0.015,
        "l2_leaf_reg": 14,
        "random_strength": 2.5,
        "bagging_temperature": 1.0,
        "iterations": 1300
    },
    {
        "name": "simple_depth2",
        "depth": 2,
        "learning_rate": 0.035,
        "l2_leaf_reg": 12,
        "random_strength": 2.0,
        "bagging_temperature": 0.9,
        "iterations": 800
    }
]

v3_models = []
v3_val_raw_preds = {}
v3_results = []

for cfg in cat_configs_v3:
    print("\nTraining config:", cfg["name"])

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=42,
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    model.fit(
        X_dev_v3,
        y_dev_v3,
        cat_features=cat_features_v3,
        eval_set=(X_val_v3, y_val_v3),
        use_best_model=True
    )

    pred = normalize_probs(model.predict_proba(X_val_v3))
    raw_loss = log_loss(y_val_v3, pred, labels=[0, 1, 2, 3])

    v3_models.append((cfg["name"], model, cfg))
    v3_val_raw_preds[cfg["name"]] = pred

    print(cfg["name"], "raw loss:", raw_loss, "best iter:", model.best_iteration_)

    v3_results.append({
        "name": cfg["name"],
        "raw_loss": raw_loss,
        "best_iter": model.best_iteration_
    })

print("\nRaw model results:")
for r in v3_results:
    print(r)


Training config: safe_depth3
0:	learn: 1.3846576	test: 1.3842858	best: 1.3842858 (0)	total: 22.5ms	remaining: 20.2s
100:	learn: 1.3292125	test: 1.3457923	best: 1.3456295 (92)	total: 2.01s	remaining: 15.9s
200:	learn: 1.3030050	test: 1.3452300	best: 1.3408386 (152)	total: 3.97s	remaining: 13.8s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 1.340838628
bestIteration = 152

Shrink model to first 153 iterations.
safe_depth3 raw loss: 1.3408386282757607 best iter: 152

Training config: balanced_depth4
0:	learn: 1.3850034	test: 1.3854567	best: 1.3854567 (0)	total: 42.1ms	remaining: 46.2s
100:	learn: 1.3180010	test: 1.3387724	best: 1.3387724 (100)	total: 4.17s	remaining: 41.2s
200:	learn: 1.2824447	test: 1.3312912	best: 1.3305643 (198)	total: 7.55s	remaining: 33.8s
300:	learn: 1.2514118	test: 1.3304606	best: 1.3290134 (289)	total: 10.9s	remaining: 29s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 1.329013353
bestIteration = 289

Shrink model to first 2

In [ ]:
# ============================================================
# VERSION 3: ENSEMBLE + CALIBRATION
# ============================================================

model_names_v3 = list(v3_val_raw_preds.keys())

ensemble_candidates = []

# Single model candidates
for name in model_names_v3:
    ensemble_candidates.append({name: 1.0})

# Simple equal ensemble
ensemble_candidates.append({name: 1.0 for name in model_names_v3})

# Weighted toward better regularized models
ensemble_candidates.append({
    "safe_depth3": 2.0,
    "balanced_depth4": 1.0,
    "strong_reg_depth4": 2.0,
    "simple_depth2": 1.0
})

ensemble_candidates.append({
    "safe_depth3": 1.0,
    "balanced_depth4": 1.0,
    "strong_reg_depth4": 3.0,
    "simple_depth2": 1.0
})

best_v3_public = {
    "loss": 999,
    "weights": None,
    "gamma": None,
    "alpha": None
}

for weights in ensemble_candidates:
    p = np.zeros_like(list(v3_val_raw_preds.values())[0])
    total_w = 0

    for name, w in weights.items():
        if name in v3_val_raw_preds:
            p += w * v3_val_raw_preds[name]
            total_w += w

    p = p / total_w
    p = normalize_probs(p)

    for gamma in np.linspace(0.65, 1.45, 41):
        pg = apply_gamma(p, gamma)

        for alpha in np.linspace(0.45, 1.00, 56):
            pf = blend_prior(pg, prior_dev_v3, alpha)
            loss = log_loss(y_val_v3, pf, labels=[0, 1, 2, 3])

            if loss < best_v3_public["loss"]:
                best_v3_public = {
                    "loss": loss,
                    "weights": weights,
                    "gamma": gamma,
                    "alpha": alpha
                }

print("BEST V3 PUBLIC VALIDATION RESULT:")
print(best_v3_public)

print("\nCompare:")
print("Old V2 best was around:", best_v2["loss"])
print("New V3 public-only best:", best_v3_public["loss"])

In [ ]:
# ============================================================
# VERSION 3: FINAL PUBLIC MODEL TRAINING
# ============================================================

X_all_v3_public = train_df_v3_public.drop(columns=drop_cols)
y_all_v3_public = train_df_v3_public["target"].map(CLASS_TO_ID).values

cat_cols_v3_final = X_all_v3_public.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v3_final:
    X_all_v3_public[c] = X_all_v3_public[c].fillna("Unknown").astype(str)

cat_features_v3_final = [X_all_v3_public.columns.get_loc(c) for c in cat_cols_v3_final]

final_public_models_v3 = {}

for name, old_model, cfg in v3_models:
    final_iters = old_model.best_iteration_ + 1 if old_model.best_iteration_ is not None else 150

    print("\nFinal training:", name, "iterations:", final_iters)

    final_model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=final_iters,
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=42,
        verbose=100
    )

    final_model.fit(
        X_all_v3_public,
        y_all_v3_public,
        cat_features=cat_features_v3_final
    )

    final_public_models_v3[name] = final_model

prior_all_v3_public = np.bincount(y_all_v3_public, minlength=4) / len(y_all_v3_public)

print("Final V3 public prior:")
print(dict(zip(CLASS_ORDER, prior_all_v3_public)))

In [ ]:
# ============================================================
# VERSION 3: PUBLIC KNOWN-TOSS PREDICTION FUNCTIONS
# ============================================================

def predict_raw_public_v3(feature_df):
    feature_df = feature_df.copy()

    for c in cat_cols_v3_final:
        if c not in feature_df.columns:
            feature_df[c] = "Unknown"
        feature_df[c] = feature_df[c].fillna("Unknown").astype(str)

    for c in X_all_v3_public.columns:
        if c not in feature_df.columns:
            feature_df[c] = np.nan

    feature_df = feature_df[X_all_v3_public.columns]

    weights = best_v3_public["weights"]
    total_w = sum(weights.values())

    p = np.zeros((len(feature_df), 4))

    for name, w in weights.items():
        p += w * final_public_models_v3[name].predict_proba(feature_df)

    p = p / total_w

    return normalize_probs(p)

def postprocess_public_v3(p):
    p = apply_gamma(p, best_v3_public["gamma"])
    p = blend_prior(p, prior_all_v3_public, best_v3_public["alpha"])
    return normalize_probs(p)

def predict_known_toss_public_v3(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(toss_winner == team_a),
        toss_decision_bat=int(toss_decision == "bat")
    )

    feature_df = pd.DataFrame([feat])
    raw = predict_raw_public_v3(feature_df)
    final = postprocess_public_v3(raw)

    return final[0]

In [ ]:
# ============================================================
# CREATE submission_v3.csv
# Public rows = V3 public model
# Private rows = V2 pre-toss model
# ============================================================

import numpy as np
import pandas as pd

pred_dict_v3 = {}

# Public rows: V3 known-toss model
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss_public_v3(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict_v3[match_id] = p

# Private rows: V2 pre-toss model
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict_v3[match_id] = p

submission_v3 = sample.copy()
submission_v3["match_id"] = submission_v3["match_id"].astype(str)

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]

for i, match_id in enumerate(submission_v3["match_id"]):
    if match_id not in pred_dict_v3:
        print("Missing:", match_id, "using prior fallback")
        p = prior_all_v3_public.copy()
    else:
        p = pred_dict_v3[match_id]

    p = normalize_probs(np.asarray(p).reshape(1, -1))[0]

    submission_v3.loc[i, "A_small"] = p[0]
    submission_v3.loc[i, "A_big"] = p[1]
    submission_v3.loc[i, "B_small"] = p[2]
    submission_v3.loc[i, "B_big"] = p[3]

assert submission_v3.shape == (53, 5)
assert submission_v3["match_id"].astype(str).tolist() == sample["match_id"].astype(str).tolist()
assert np.all(submission_v3[CLASS_ORDER].values >= 0)
assert np.all(submission_v3[CLASS_ORDER].values <= 1)
assert np.allclose(submission_v3[CLASS_ORDER].sum(axis=1), 1.0, atol=1e-6)

submission_v3.to_csv("submission_v3.csv", index=False)

print("✅ submission_v3.csv saved")
print(submission_v3.shape)
print(submission_v3[CLASS_ORDER].mean())
display(submission_v3.head())
display(submission_v3.tail())

#V4 STARTS HERE

# Great progress: V3 moved from 1.365 → 1.319, which is a real improvement and now close to competition range. To push nearer 1.25, the next safe upgrade is role-specific features: how Team A performs when batting first and how Team B performs while chasing.

Thought for 16s

Yes — V3 is valid and no obvious leakage.

Your result is strong improvement:

# V1 best: 1.3876
#V2 best: 1.3651
#V3 best: 1.3190

Now to push closer to 1.25–1.30, the next best safe upgrade is:

Role-specific model
Team A in public matches is batting first. Team B is chasing.
So we add features like:

Team A recent record when batting first
Team A recent big-win rate while batting first
Team B recent chasing win rate
Team B recent chasing big-win rate
Team A defending tendency
Team B chasing tendency

This is not leakage because we calculate only from matches before the current match date.

# V4 CELL 1 — Role-specific history

# Run this after V3/V2 cells are already available.

In [ ]:
# ============================================================
# VERSION 4: ROLE-SPECIFIC HISTORY
# Bat-first and chasing features for public known-toss matches
# ============================================================

def build_role_history(ball_df, match_df):
    rows = []

    for _, r in match_df.iterrows():
        match_id = r["Match ID"]
        date = r["Date"]
        venue = r["Venue"]
        city = r["city"]

        bat_first = r["Bat First"]
        bat_second = r["Bat Second"]
        winner = r["match_won_by"]
        label = r["target_label"]

        if pd.isna(label):
            continue

        inn1_runs = r["inn1_innings_runs"]
        inn2_runs = r["inn2_innings_runs"]
        inn1_wkts = r["inn1_innings_wickets"]
        inn2_wkts = r["inn2_innings_wickets"]

        # Team batting first role
        rows.append({
            "Match ID": match_id,
            "Date": date,
            "Venue": venue,
            "city": city,
            "team": bat_first,
            "opponent": bat_second,
            "role": "bat_first",
            "runs_scored": inn1_runs,
            "runs_conceded": inn2_runs,
            "wickets_lost": inn1_wkts,
            "wickets_taken": inn2_wkts,
            "won": int(winner == bat_first),
            "big_win": int(label == "A_big"),
            "small_win": int(label == "A_small"),
            "big_loss": int(label == "B_big"),
            "small_loss": int(label == "B_small"),
        })

        # Team chasing role
        rows.append({
            "Match ID": match_id,
            "Date": date,
            "Venue": venue,
            "city": city,
            "team": bat_second,
            "opponent": bat_first,
            "role": "chase",
            "runs_scored": inn2_runs,
            "runs_conceded": inn1_runs,
            "wickets_lost": inn2_wkts,
            "wickets_taken": inn1_wkts,
            "won": int(winner == bat_second),
            "big_win": int(label == "B_big"),
            "small_win": int(label == "B_small"),
            "big_loss": int(label == "A_big"),
            "small_loss": int(label == "A_small"),
        })

    role_history = pd.DataFrame(rows)
    role_history = role_history.sort_values(["Date", "Match ID"]).reset_index(drop=True)
    return role_history


role_history = build_role_history(train, labelable_matches)

print(role_history.shape)
print(role_history["role"].value_counts())
role_history.head()

In [ ]:
# ============================================================
# VERSION 4: ROLE FEATURE SUMMARIES
# ============================================================

def summarize_role_window(h, prefix, window):
    recent = h.tail(window)
    out = {}

    out[f"{prefix}_role_count_w{window}"] = len(recent)

    if len(recent) == 0:
        return out

    out[f"{prefix}_role_win_rate_w{window}"] = recent["won"].mean()
    out[f"{prefix}_role_big_win_rate_w{window}"] = recent["big_win"].mean()
    out[f"{prefix}_role_small_win_rate_w{window}"] = recent["small_win"].mean()
    out[f"{prefix}_role_big_loss_rate_w{window}"] = recent["big_loss"].mean()
    out[f"{prefix}_role_small_loss_rate_w{window}"] = recent["small_loss"].mean()

    out[f"{prefix}_role_avg_runs_scored_w{window}"] = recent["runs_scored"].mean()
    out[f"{prefix}_role_avg_runs_conceded_w{window}"] = recent["runs_conceded"].mean()
    out[f"{prefix}_role_avg_wickets_lost_w{window}"] = recent["wickets_lost"].mean()
    out[f"{prefix}_role_avg_wickets_taken_w{window}"] = recent["wickets_taken"].mean()

    out[f"{prefix}_role_net_runs_w{window}"] = (
        recent["runs_scored"].mean() - recent["runs_conceded"].mean()
    )

    out[f"{prefix}_role_wicket_diff_w{window}"] = (
        recent["wickets_taken"].mean() - recent["wickets_lost"].mean()
    )

    return out


def summarize_team_role(prior_role_history, team, role, prefix, date):
    h = prior_role_history[
        (prior_role_history["team"] == team) &
        (prior_role_history["role"] == role)
    ].sort_values("Date")

    out = {}

    out[f"{prefix}_role_total_seen"] = len(h)

    if len(h) > 0:
        out[f"{prefix}_role_days_since_last"] = (date - h["Date"].max()).days
    else:
        out[f"{prefix}_role_days_since_last"] = np.nan

    for window in [3, 5, 10, 20]:
        out.update(summarize_role_window(h, prefix, window))

    return out

In [ ]:
# ============================================================
# VERSION 4: FEATURE ROW WITH ROLE-SPECIFIC FEATURES
# ============================================================

def make_feature_row_v4(date, team_a, team_b, venue, city, toss_winner_is_A, toss_decision_bat):
    row = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=toss_winner_is_A,
        toss_decision_bat=toss_decision_bat
    )

    prior_role_history = role_history[role_history["Date"] < date].copy()

    # For public/original perspective:
    # Team A = Bat First, Team B = Bat Second / Chase
    row.update(
        summarize_team_role(
            prior_role_history=prior_role_history,
            team=team_a,
            role="bat_first",
            prefix="A_batfirst",
            date=date
        )
    )

    row.update(
        summarize_team_role(
            prior_role_history=prior_role_history,
            team=team_b,
            role="chase",
            prefix="B_chase",
            date=date
        )
    )

    # Opposite role tendencies are also useful
    row.update(
        summarize_team_role(
            prior_role_history=prior_role_history,
            team=team_a,
            role="chase",
            prefix="A_chase",
            date=date
        )
    )

    row.update(
        summarize_team_role(
            prior_role_history=prior_role_history,
            team=team_b,
            role="bat_first",
            prefix="B_batfirst",
            date=date
        )
    )

    # Difference features between public match roles
    for c in list(row.keys()):
        if c.startswith("A_batfirst_"):
            b_col = "B_chase_" + c.replace("A_batfirst_", "")
            if b_col in row:
                row["diff_public_role_" + c.replace("A_batfirst_", "")] = row[c] - row[b_col]

    return row

In [ ]:
# ============================================================
# VERSION 4: PUBLIC ORIGINAL-ONLY TRAINING DATA
# ============================================================

train_rows_v4_public = []

for _, r in labelable_matches.iterrows():
    date = r["Date"]
    venue = r["Venue"]
    city = r["city"]

    team_a = r["Bat First"]
    team_b = r["Bat Second"]

    feat = make_feature_row_v4(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(r["toss_winner"] == team_a),
        toss_decision_bat=int(r["toss_decision"] == "bat")
    )

    feat["Date"] = date
    feat["match_id"] = r["Match ID"]
    feat["target"] = r["target_label"]

    train_rows_v4_public.append(feat)

train_df_v4_public = pd.DataFrame(train_rows_v4_public)
train_df_v4_public = train_df_v4_public.sort_values("Date").reset_index(drop=True)

print("V4 public train shape:", train_df_v4_public.shape)
print(train_df_v4_public["target"].value_counts(normalize=True))
print(train_df_v4_public["target"].value_counts())

In [ ]:
# ============================================================
# VERSION 4: TIME VALIDATION
# ============================================================

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_ORDER)}

drop_cols = ["target", "Date", "match_id"]

val_mask_v4 = train_df_v4_public["Date"] >= pd.Timestamp("2025-01-01")

if val_mask_v4.sum() < 30:
    cutoff_idx = int(len(train_df_v4_public) * 0.85)
    cutoff_date = train_df_v4_public["Date"].iloc[cutoff_idx]
    val_mask_v4 = train_df_v4_public["Date"] >= cutoff_date

dev_df_v4 = train_df_v4_public[~val_mask_v4].copy()
val_df_v4 = train_df_v4_public[val_mask_v4].copy()

X_dev_v4 = dev_df_v4.drop(columns=drop_cols)
y_dev_v4 = dev_df_v4["target"].map(CLASS_TO_ID).values

X_val_v4 = val_df_v4.drop(columns=drop_cols)
y_val_v4 = val_df_v4["target"].map(CLASS_TO_ID).values

cat_cols_v4 = X_dev_v4.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v4:
    X_dev_v4[c] = X_dev_v4[c].fillna("Unknown").astype(str)
    X_val_v4[c] = X_val_v4[c].fillna("Unknown").astype(str)

cat_features_v4 = [X_dev_v4.columns.get_loc(c) for c in cat_cols_v4]

print("V4 Dev:", dev_df_v4.shape, dev_df_v4["Date"].min(), dev_df_v4["Date"].max())
print("V4 Val:", val_df_v4.shape, val_df_v4["Date"].min(), val_df_v4["Date"].max())
print("V4 Features:", X_dev_v4.shape[1])
print("V4 Categorical:", cat_cols_v4)
print("V4 Val target distribution:")
print(val_df_v4["target"].value_counts())

In [ ]:
# ============================================================
# VERSION 4: CATBOOST PARAMETER SEARCH
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
import numpy as np

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

prior_dev_v4 = np.bincount(y_dev_v4, minlength=4) / len(y_dev_v4)

cat_configs_v4 = [
    {
        "name": "v4_balanced_depth4",
        "depth": 4,
        "learning_rate": 0.018,
        "l2_leaf_reg": 8,
        "random_strength": 1.5,
        "bagging_temperature": 0.7,
        "iterations": 1200
    },
    {
        "name": "v4_safe_depth3",
        "depth": 3,
        "learning_rate": 0.023,
        "l2_leaf_reg": 10,
        "random_strength": 2.0,
        "bagging_temperature": 0.8,
        "iterations": 1000
    },
    {
        "name": "v4_simple_depth2",
        "depth": 2,
        "learning_rate": 0.035,
        "l2_leaf_reg": 12,
        "random_strength": 2.0,
        "bagging_temperature": 0.9,
        "iterations": 900
    },
    {
        "name": "v4_more_reg_depth4",
        "depth": 4,
        "learning_rate": 0.015,
        "l2_leaf_reg": 16,
        "random_strength": 2.5,
        "bagging_temperature": 1.0,
        "iterations": 1400
    }
]

v4_models = []
v4_val_raw_preds = {}
v4_results = []

for cfg in cat_configs_v4:
    print("\nTraining config:", cfg["name"])

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=42,
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    model.fit(
        X_dev_v4,
        y_dev_v4,
        cat_features=cat_features_v4,
        eval_set=(X_val_v4, y_val_v4),
        use_best_model=True
    )

    pred = normalize_probs(model.predict_proba(X_val_v4))
    raw_loss = log_loss(y_val_v4, pred, labels=[0, 1, 2, 3])

    v4_models.append((cfg["name"], model, cfg))
    v4_val_raw_preds[cfg["name"]] = pred

    print(cfg["name"], "raw loss:", raw_loss, "best iter:", model.best_iteration_)

    v4_results.append({
        "name": cfg["name"],
        "raw_loss": raw_loss,
        "best_iter": model.best_iteration_
    })

print("\nV4 Raw model results:")
for r in v4_results:
    print(r)

In [ ]:
# ============================================================
# VERSION 4: ENSEMBLE + CALIBRATION
# ============================================================

model_names_v4 = list(v4_val_raw_preds.keys())

ensemble_candidates_v4 = []

# Single models
for name in model_names_v4:
    ensemble_candidates_v4.append({name: 1.0})

# Equal ensemble
ensemble_candidates_v4.append({name: 1.0 for name in model_names_v4})

# Weighted options
ensemble_candidates_v4.append({
    "v4_balanced_depth4": 2.0,
    "v4_safe_depth3": 1.0,
    "v4_simple_depth2": 1.0,
    "v4_more_reg_depth4": 1.0
})

ensemble_candidates_v4.append({
    "v4_balanced_depth4": 3.0,
    "v4_safe_depth3": 1.0,
    "v4_simple_depth2": 1.0,
    "v4_more_reg_depth4": 1.0
})

best_v4_public = {
    "loss": 999,
    "weights": None,
    "gamma": None,
    "alpha": None
}

for weights in ensemble_candidates_v4:
    p = np.zeros_like(list(v4_val_raw_preds.values())[0])
    total_w = 0

    for name, w in weights.items():
        if name in v4_val_raw_preds:
            p += w * v4_val_raw_preds[name]
            total_w += w

    p = normalize_probs(p / total_w)

    for gamma in np.linspace(0.70, 1.70, 51):
        pg = apply_gamma(p, gamma)

        for alpha in np.linspace(0.50, 1.00, 51):
            pf = blend_prior(pg, prior_dev_v4, alpha)
            loss = log_loss(y_val_v4, pf, labels=[0, 1, 2, 3])

            if loss < best_v4_public["loss"]:
                best_v4_public = {
                    "loss": loss,
                    "weights": weights,
                    "gamma": gamma,
                    "alpha": alpha
                }

print("BEST V4 PUBLIC VALIDATION RESULT:")
print(best_v4_public)

print("\nCompare:")
print("V2 best:", best_v2["loss"])
print("V3 public best:", best_v3_public["loss"])
print("V4 public best:", best_v4_public["loss"])

In [ ]:
# ============================================================
# FIX FOR V4: FINAL PUBLIC MODEL TRAINING
# Run this BEFORE creating submission_v4.csv
# ============================================================

from catboost import CatBoostClassifier
import numpy as np

drop_cols = ["target", "Date", "match_id"]

X_all_v4_public = train_df_v4_public.drop(columns=drop_cols)
y_all_v4_public = train_df_v4_public["target"].map(CLASS_TO_ID).values

cat_cols_v4_final = X_all_v4_public.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v4_final:
    X_all_v4_public[c] = X_all_v4_public[c].fillna("Unknown").astype(str)

cat_features_v4_final = [X_all_v4_public.columns.get_loc(c) for c in cat_cols_v4_final]

final_public_models_v4 = {}

for name, old_model, cfg in v4_models:
    final_iters = old_model.best_iteration_ + 1 if old_model.best_iteration_ is not None else 150

    print("Final V4 training:", name, "iterations:", final_iters)

    final_model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=final_iters,
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=42,
        verbose=100
    )

    final_model.fit(
        X_all_v4_public,
        y_all_v4_public,
        cat_features=cat_features_v4_final
    )

    final_public_models_v4[name] = final_model

prior_all_v4_public = np.bincount(y_all_v4_public, minlength=4) / len(y_all_v4_public)

print("✅ V4 final variables created successfully")
print("cat_cols_v4_final:", cat_cols_v4_final)
print("Final V4 public prior:")
print(dict(zip(CLASS_ORDER, prior_all_v4_public)))

In [ ]:
# ============================================================
# V4 FINAL: CREATE + CHECK + DOWNLOAD submission_v4.csv
# Public rows = V4 known-toss model
# Private rows = V2 pre-toss model
# ============================================================

import numpy as np
import pandas as pd
from google.colab import files
from catboost import CatBoostClassifier

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]
drop_cols = ["target", "Date", "match_id"]

# ------------------------------------------------------------
# 1. Create V4 final model variables if not already created
# ------------------------------------------------------------
if "cat_cols_v4_final" not in globals():

    print("V4 final variables not found. Creating them now...")

    X_all_v4_public = train_df_v4_public.drop(columns=drop_cols)
    y_all_v4_public = train_df_v4_public["target"].map(CLASS_TO_ID).values

    cat_cols_v4_final = X_all_v4_public.select_dtypes(include=["object"]).columns.tolist()

    for c in cat_cols_v4_final:
        X_all_v4_public[c] = X_all_v4_public[c].fillna("Unknown").astype(str)

    cat_features_v4_final = [X_all_v4_public.columns.get_loc(c) for c in cat_cols_v4_final]

    final_public_models_v4 = {}

    for name, old_model, cfg in v4_models:
        final_iters = old_model.best_iteration_ + 1 if old_model.best_iteration_ is not None else 150

        print("Final V4 training:", name, "iterations:", final_iters)

        final_model = CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric="MultiClass",
            iterations=final_iters,
            depth=cfg["depth"],
            learning_rate=cfg["learning_rate"],
            l2_leaf_reg=cfg["l2_leaf_reg"],
            random_strength=cfg["random_strength"],
            bagging_temperature=cfg["bagging_temperature"],
            border_count=128,
            random_seed=42,
            verbose=100
        )

        final_model.fit(
            X_all_v4_public,
            y_all_v4_public,
            cat_features=cat_features_v4_final
        )

        final_public_models_v4[name] = final_model

    prior_all_v4_public = np.bincount(y_all_v4_public, minlength=4) / len(y_all_v4_public)

    print("✅ V4 final models created")

else:
    print("✅ V4 final variables already exist")


# ------------------------------------------------------------
# 2. V4 prediction helper functions
# ------------------------------------------------------------
def predict_raw_public_v4(feature_df):
    feature_df = feature_df.copy()

    for c in cat_cols_v4_final:
        if c not in feature_df.columns:
            feature_df[c] = "Unknown"
        feature_df[c] = feature_df[c].fillna("Unknown").astype(str)

    for c in X_all_v4_public.columns:
        if c not in feature_df.columns:
            feature_df[c] = np.nan

    feature_df = feature_df[X_all_v4_public.columns]

    weights = best_v4_public["weights"]
    total_w = sum(weights.values())

    p = np.zeros((len(feature_df), 4))

    for name, w in weights.items():
        p += w * final_public_models_v4[name].predict_proba(feature_df)

    p = p / total_w
    return normalize_probs(p)


def postprocess_public_v4(p):
    p = apply_gamma(p, best_v4_public["gamma"])
    p = blend_prior(p, prior_all_v4_public, best_v4_public["alpha"])
    return normalize_probs(p)


def predict_known_toss_public_v4(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    feat = make_feature_row_v4(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(toss_winner == team_a),
        toss_decision_bat=int(toss_decision == "bat")
    )

    feature_df = pd.DataFrame([feat])
    raw = predict_raw_public_v4(feature_df)
    final = postprocess_public_v4(raw)

    return final[0]


# ------------------------------------------------------------
# 3. Create predictions
# ------------------------------------------------------------
pred_dict_v4 = {}

# Public rows: V4 known-toss model
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss_public_v4(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict_v4[match_id] = p

# Private rows: V2 pre-toss model
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict_v4[match_id] = p


# ------------------------------------------------------------
# 4. Build submission_v4.csv
# ------------------------------------------------------------
submission_v4 = sample.copy()
submission_v4["match_id"] = submission_v4["match_id"].astype(str)

for i, match_id in enumerate(submission_v4["match_id"]):
    if match_id not in pred_dict_v4:
        print("Missing:", match_id, "using V4 prior fallback")
        p = prior_all_v4_public.copy()
    else:
        p = pred_dict_v4[match_id]

    p = normalize_probs(np.asarray(p).reshape(1, -1))[0]

    submission_v4.loc[i, "A_small"] = p[0]
    submission_v4.loc[i, "A_big"] = p[1]
    submission_v4.loc[i, "B_small"] = p[2]
    submission_v4.loc[i, "B_big"] = p[3]

# ------------------------------------------------------------
# 5. Final checks
# ------------------------------------------------------------
assert submission_v4.shape == (53, 5), f"Wrong shape: {submission_v4.shape}"
assert submission_v4.columns.tolist() == ["match_id", "A_small", "A_big", "B_small", "B_big"]
assert submission_v4["match_id"].astype(str).tolist() == sample["match_id"].astype(str).tolist()
assert submission_v4[CLASS_ORDER].isna().sum().sum() == 0
assert np.isfinite(submission_v4[CLASS_ORDER].values).all()
assert np.all(submission_v4[CLASS_ORDER].values >= 0)
assert np.all(submission_v4[CLASS_ORDER].values <= 1)
assert np.allclose(submission_v4[CLASS_ORDER].sum(axis=1), 1.0, atol=1e-6)

submission_v4.to_csv("submission_v4.csv", index=False)

print("✅ submission_v4.csv saved successfully")
print("Rows:", submission_v4.shape)
print("Probability row sum min:", submission_v4[CLASS_ORDER].sum(axis=1).min())
print("Probability row sum max:", submission_v4[CLASS_ORDER].sum(axis=1).max())

print("\nAverage probabilities:")
print(submission_v4[CLASS_ORDER].mean())

display(submission_v4.head())
display(submission_v4.tail())

# ------------------------------------------------------------
# 6. Download
# ------------------------------------------------------------
files.download("submission_v4.csv")

#V5 STARTS HERE


V4 is worse than V3, so we should not use V4 for submission. One stronger safe attempt is not more features, but better validation/control: remove noisy role features, keep V3 features, add recency-weighting + model averaging across random seeds.

Thought for a couple of seconds

Yes, V4 result says clearly:

V3 public best: 1.3190
V4 public best: 1.3311

So V4 is worse. Do not submit V4.

Now we try one final strong version: V5 = V3 features + multi-seed CatBoost ensemble + sample recency weights.

Why this can improve:

V3 features were better than V4.
V4 added too many noisy role features.
Instead of adding more noisy columns, we stabilize CatBoost using:
multiple random seeds
recency sample weights
best V3 model structure
sharper probability calibration
V5 CELL 1 — Use V3 data, add recency sample weights

Run this after V3 is already available.

In [ ]:
# ============================================================
# VERSION 5: V3 FEATURES + RECENCY WEIGHTS + MULTI-SEED CATBOOST
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
import numpy as np
import pandas as pd

# We use V3 data because V3 beat V4
X_dev_v5 = X_dev_v3.copy()
X_val_v5 = X_val_v3.copy()
y_dev_v5 = y_dev_v3.copy()
y_val_v5 = y_val_v3.copy()

cat_cols_v5 = cat_cols_v3.copy()
cat_features_v5 = cat_features_v3.copy()

# Recency weights: newer matches matter more
dev_years = dev_df_v3["Date"].dt.year.values

sample_weight_v5 = np.ones(len(dev_df_v3))

sample_weight_v5[dev_years >= 2024] = 1.60
sample_weight_v5[dev_years == 2023] = 1.35
sample_weight_v5[dev_years == 2022] = 1.20
sample_weight_v5[dev_years <= 2015] = 0.75

print("Sample weight min:", sample_weight_v5.min())
print("Sample weight max:", sample_weight_v5.max())
print("X_dev_v5:", X_dev_v5.shape)
print("X_val_v5:", X_val_v5.shape)

In [ ]:
# ============================================================
# VERSION 5: MULTI-SEED CATBOOST TRAINING
# ============================================================

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

prior_dev_v5 = np.bincount(y_dev_v5, minlength=4) / len(y_dev_v5)

v5_configs = [
    {
        "name": "v5_balanced_depth4_seed42",
        "depth": 4,
        "learning_rate": 0.018,
        "l2_leaf_reg": 8,
        "random_strength": 1.5,
        "bagging_temperature": 0.7,
        "iterations": 1000,
        "seed": 42
    },
    {
        "name": "v5_balanced_depth4_seed7",
        "depth": 4,
        "learning_rate": 0.018,
        "l2_leaf_reg": 8,
        "random_strength": 1.5,
        "bagging_temperature": 0.7,
        "iterations": 1000,
        "seed": 7
    },
    {
        "name": "v5_balanced_depth4_seed2026",
        "depth": 4,
        "learning_rate": 0.018,
        "l2_leaf_reg": 8,
        "random_strength": 1.5,
        "bagging_temperature": 0.7,
        "iterations": 1000,
        "seed": 2026
    },
    {
        "name": "v5_simple_depth2_seed42",
        "depth": 2,
        "learning_rate": 0.035,
        "l2_leaf_reg": 12,
        "random_strength": 2.0,
        "bagging_temperature": 0.9,
        "iterations": 800,
        "seed": 42
    },
    {
        "name": "v5_safe_depth3_seed99",
        "depth": 3,
        "learning_rate": 0.025,
        "l2_leaf_reg": 10,
        "random_strength": 2.0,
        "bagging_temperature": 0.8,
        "iterations": 900,
        "seed": 99
    }
]

v5_models = []
v5_val_raw_preds = {}
v5_results = []

for cfg in v5_configs:
    print("\nTraining:", cfg["name"])

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"],
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    model.fit(
        X_dev_v5,
        y_dev_v5,
        cat_features=cat_features_v5,
        sample_weight=sample_weight_v5,
        eval_set=(X_val_v5, y_val_v5),
        use_best_model=True
    )

    pred = normalize_probs(model.predict_proba(X_val_v5))
    raw_loss = log_loss(y_val_v5, pred, labels=[0, 1, 2, 3])

    v5_models.append((cfg["name"], model, cfg))
    v5_val_raw_preds[cfg["name"]] = pred

    result = {
        "name": cfg["name"],
        "raw_loss": raw_loss,
        "best_iter": model.best_iteration_
    }

    v5_results.append(result)
    print(result)

print("\nV5 raw results:")
for r in v5_results:
    print(r)

In [ ]:
# ============================================================
# VERSION 5: ENSEMBLE + CALIBRATION SEARCH
# ============================================================

model_names_v5 = list(v5_val_raw_preds.keys())

ensemble_candidates_v5 = []

# Single models
for name in model_names_v5:
    ensemble_candidates_v5.append({name: 1.0})

# Equal ensemble of all
ensemble_candidates_v5.append({name: 1.0 for name in model_names_v5})

# Balanced depth4 seeds only
ensemble_candidates_v5.append({
    "v5_balanced_depth4_seed42": 1.0,
    "v5_balanced_depth4_seed7": 1.0,
    "v5_balanced_depth4_seed2026": 1.0
})

# Weighted toward best V3-like model family
ensemble_candidates_v5.append({
    "v5_balanced_depth4_seed42": 2.0,
    "v5_balanced_depth4_seed7": 2.0,
    "v5_balanced_depth4_seed2026": 2.0,
    "v5_simple_depth2_seed42": 1.0,
    "v5_safe_depth3_seed99": 1.0
})

# Safer mixed ensemble
ensemble_candidates_v5.append({
    "v5_balanced_depth4_seed42": 2.0,
    "v5_simple_depth2_seed42": 1.0,
    "v5_safe_depth3_seed99": 1.0
})

best_v5_public = {
    "loss": 999,
    "weights": None,
    "gamma": None,
    "alpha": None
}

for weights in ensemble_candidates_v5:
    p = np.zeros_like(list(v5_val_raw_preds.values())[0])
    total_w = 0

    for name, w in weights.items():
        if name in v5_val_raw_preds:
            p += w * v5_val_raw_preds[name]
            total_w += w

    p = normalize_probs(p / total_w)

    for gamma in np.linspace(0.80, 1.80, 51):
        pg = apply_gamma(p, gamma)

        for alpha in np.linspace(0.70, 1.00, 31):
            pf = blend_prior(pg, prior_dev_v5, alpha)
            loss = log_loss(y_val_v5, pf, labels=[0, 1, 2, 3])

            if loss < best_v5_public["loss"]:
                best_v5_public = {
                    "loss": loss,
                    "weights": weights,
                    "gamma": gamma,
                    "alpha": alpha
                }

print("BEST V5 PUBLIC VALIDATION RESULT:")
print(best_v5_public)

print("\nCompare:")
print("V2 best:", best_v2["loss"])
print("V3 public best:", best_v3_public["loss"])
print("V4 public best:", best_v4_public["loss"])
print("V5 public best:", best_v5_public["loss"])

In [48]:
# ============================================================
# VERSION 5: FINAL PUBLIC MODEL TRAINING
# ============================================================

X_all_v5_public = train_df_v3_public.drop(columns=drop_cols)
y_all_v5_public = train_df_v3_public["target"].map(CLASS_TO_ID).values

cat_cols_v5_final = X_all_v5_public.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v5_final:
    X_all_v5_public[c] = X_all_v5_public[c].fillna("Unknown").astype(str)

cat_features_v5_final = [X_all_v5_public.columns.get_loc(c) for c in cat_cols_v5_final]

# Final recency weights on all public/original training rows
all_years_v5 = train_df_v3_public["Date"].dt.year.values
sample_weight_all_v5 = np.ones(len(train_df_v3_public))

sample_weight_all_v5[all_years_v5 >= 2024] = 1.60
sample_weight_all_v5[all_years_v5 == 2023] = 1.35
sample_weight_all_v5[all_years_v5 == 2022] = 1.20
sample_weight_all_v5[all_years_v5 <= 2015] = 0.75

final_public_models_v5 = {}

for name, old_model, cfg in v5_models:
    final_iters = old_model.best_iteration_ + 1 if old_model.best_iteration_ is not None else 150

    print("\nFinal V5 training:", name, "iterations:", final_iters)

    final_model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=final_iters,
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"],
        verbose=100
    )

    final_model.fit(
        X_all_v5_public,
        y_all_v5_public,
        cat_features=cat_features_v5_final,
        sample_weight=sample_weight_all_v5
    )

    final_public_models_v5[name] = final_model

prior_all_v5_public = np.bincount(y_all_v5_public, minlength=4) / len(y_all_v5_public)

print("Final V5 public prior:")
print(dict(zip(CLASS_ORDER, prior_all_v5_public)))


Final V5 training: v5_balanced_depth4_seed42 iterations: 275
0:	learn: 1.3847848	total: 41.4ms	remaining: 11.3s
100:	learn: 1.3198182	total: 4.09s	remaining: 7.05s
200:	learn: 1.2839842	total: 7.36s	remaining: 2.71s
274:	learn: 1.2585472	total: 9.89s	remaining: 0us

Final V5 training: v5_balanced_depth4_seed7 iterations: 237
0:	learn: 1.3850331	total: 35.5ms	remaining: 8.37s
100:	learn: 1.3213285	total: 4.07s	remaining: 5.48s
200:	learn: 1.2831578	total: 7.42s	remaining: 1.33s
236:	learn: 1.2708424	total: 8.62s	remaining: 0us

Final V5 training: v5_balanced_depth4_seed2026 iterations: 296
0:	learn: 1.3851544	total: 32.4ms	remaining: 9.55s
100:	learn: 1.3184584	total: 3.32s	remaining: 6.41s
200:	learn: 1.2795550	total: 7.32s	remaining: 3.46s
295:	learn: 1.2480724	total: 10.5s	remaining: 0us

Final V5 training: v5_simple_depth2_seed42 iterations: 129
0:	learn: 1.3840182	total: 13ms	remaining: 1.67s
100:	learn: 1.3327723	total: 967ms	remaining: 268ms
128:	learn: 1.3276453	total: 1.25s	re

In [49]:
# ============================================================
# VERSION 5: PUBLIC KNOWN-TOSS PREDICTION FUNCTIONS
# ============================================================

def predict_raw_public_v5(feature_df):
    feature_df = feature_df.copy()

    for c in cat_cols_v5_final:
        if c not in feature_df.columns:
            feature_df[c] = "Unknown"
        feature_df[c] = feature_df[c].fillna("Unknown").astype(str)

    for c in X_all_v5_public.columns:
        if c not in feature_df.columns:
            feature_df[c] = np.nan

    feature_df = feature_df[X_all_v5_public.columns]

    weights = best_v5_public["weights"]
    total_w = sum(weights.values())

    p = np.zeros((len(feature_df), 4))

    for name, w in weights.items():
        p += w * final_public_models_v5[name].predict_proba(feature_df)

    p = p / total_w

    return normalize_probs(p)

def postprocess_public_v5(p):
    p = apply_gamma(p, best_v5_public["gamma"])
    p = blend_prior(p, prior_all_v5_public, best_v5_public["alpha"])
    return normalize_probs(p)

def predict_known_toss_public_v5(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(toss_winner == team_a),
        toss_decision_bat=int(toss_decision == "bat")
    )

    feature_df = pd.DataFrame([feat])
    raw = predict_raw_public_v5(feature_df)
    final = postprocess_public_v5(raw)

    return final[0]

In [50]:
# ============================================================
# CREATE submission_v5.csv AGAIN
# Public rows = V5 known-toss model
# Private rows = V2 symmetric pre-toss model
# ============================================================

import numpy as np
import pandas as pd
import os

pred_dict_v5 = {}

# Public rows: V5 public known-toss model
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss_public_v5(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict_v5[match_id] = p

# Private rows: V2 pre-toss marginalization
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict_v5[match_id] = p

submission_v5 = sample.copy()
submission_v5["match_id"] = submission_v5["match_id"].astype(str)

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]

for i, match_id in enumerate(submission_v5["match_id"]):
    if match_id not in pred_dict_v5:
        print("Missing prediction for:", match_id, "using V5 prior fallback")
        p = prior_all_v5_public.copy()
    else:
        p = pred_dict_v5[match_id]

    p = normalize_probs(np.asarray(p).reshape(1, -1))[0]

    submission_v5.loc[i, "A_small"] = p[0]
    submission_v5.loc[i, "A_big"] = p[1]
    submission_v5.loc[i, "B_small"] = p[2]
    submission_v5.loc[i, "B_big"] = p[3]

# Hard checks
assert submission_v5.shape == (53, 5), f"Wrong shape: {submission_v5.shape}"
assert submission_v5["match_id"].astype(str).tolist() == sample["match_id"].astype(str).tolist()
assert np.all(submission_v5[CLASS_ORDER].values >= 0)
assert np.all(submission_v5[CLASS_ORDER].values <= 1)
assert np.allclose(submission_v5[CLASS_ORDER].sum(axis=1).values, 1.0, atol=1e-6)

submission_v5.to_csv("submission_v5.csv", index=False)

print("✅ submission_v5.csv saved successfully")
print("Rows:", submission_v5.shape)
print("Probability row sum min:", submission_v5[CLASS_ORDER].sum(axis=1).min())
print("Probability row sum max:", submission_v5[CLASS_ORDER].sum(axis=1).max())

print("\nAverage probabilities:")
print(submission_v5[CLASS_ORDER].mean())

display(submission_v5.head())
display(submission_v5.tail())

✅ submission_v5.csv saved successfully
Rows: (53, 5)
Probability row sum min: 0.9999999999999998
Probability row sum max: 1.0

Average probabilities:
A_small    0.191508
A_big      0.231104
B_small    0.143207
B_big      0.434182
dtype: float64


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.190991,0.273279,0.121578,0.414153
1,1473489,0.188973,0.255739,0.116160,0.439127
2,1473490,0.219398,0.248390,0.106612,0.425600
3,1473491,0.193406,0.236827,0.116266,0.453501
4,1473492,0.193577,0.210344,0.138537,0.457542


,match_id,A_small,A_big,B_small,B_big
48,M_2026_T01,0.195267,0.293354,0.203494,0.307885
49,M_2026_T02,0.198750,0.297648,0.198525,0.305077
50,M_2026_T03,0.196740,0.300049,0.200914,0.302297
51,M_2026_T04,0.204248,0.308299,0.193001,0.294452
52,M_2026_T05,0.202115,0.301531,0.197217,0.299138


#V6 STARTS HERE


One more strong safe attempt makes sense, but we should change the modeling idea, not just add random features. V6 will use a two-stage model: first predict winner side, then predict big/small margin, then combine with V5.

Thought for 9s

Yes, let’s try one more serious method to break 1.30.

Your current best:

V5 public best = 1.317216

Now we try V6: Two-stage decomposition model.

Instead of directly predicting 4 classes, we split the problem into:

Model 1: Who wins?        A vs B
Model 2: Margin size?     Small vs Big

Then combine:

A_small = P(A wins) × P(small)
A_big   = P(A wins) × P(big)
B_small = P(B wins) × P(small)
B_big   = P(B wins) × P(big)

Then we blend this with your best V5 multiclass model.

This is still leakage-safe because it uses the same V3 public training data and same past-only features.

In [51]:
# ============================================================
# VERSION 6: TWO-STAGE MODEL
# Stage 1: Winner side A/B
# Stage 2: Margin size Small/Big
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
import numpy as np
import pandas as pd

# Use V3 public/original-only data because V3/V5 beat V4
train_df_v6_public = train_df_v3_public.copy()

def label_to_side(label):
    return 1 if str(label).startswith("A_") else 0
    # 1 = A wins, 0 = B wins

def label_to_big(label):
    return 1 if str(label).endswith("_big") else 0
    # 1 = big, 0 = small

train_df_v6_public["target_side_A"] = train_df_v6_public["target"].apply(label_to_side)
train_df_v6_public["target_big"] = train_df_v6_public["target"].apply(label_to_big)

print(train_df_v6_public[["target", "target_side_A", "target_big"]].head())
print("\nSide distribution:")
print(train_df_v6_public["target_side_A"].value_counts(normalize=True))
print("\nBig/small distribution:")
print(train_df_v6_public["target_big"].value_counts(normalize=True))

    target  target_side_A  target_big
0    A_big              1           1
1    A_big              1           1
2    B_big              0           1
3  B_small              0           0
4  B_small              0           0

Side distribution:
target_side_A
0    0.543594
1    0.456406
Name: proportion, dtype: float64

Big/small distribution:
target_big
1    0.599644
0    0.400356
Name: proportion, dtype: float64


In [52]:
# ============================================================
# VERSION 6: TIME VALIDATION SPLIT
# ============================================================

drop_cols_v6 = ["target", "target_side_A", "target_big", "Date", "match_id"]

val_mask_v6 = train_df_v6_public["Date"] >= pd.Timestamp("2025-01-01")

if val_mask_v6.sum() < 30:
    cutoff_idx = int(len(train_df_v6_public) * 0.85)
    cutoff_date = train_df_v6_public["Date"].iloc[cutoff_idx]
    val_mask_v6 = train_df_v6_public["Date"] >= cutoff_date

dev_df_v6 = train_df_v6_public[~val_mask_v6].copy()
val_df_v6 = train_df_v6_public[val_mask_v6].copy()

X_dev_v6 = dev_df_v6.drop(columns=drop_cols_v6)
X_val_v6 = val_df_v6.drop(columns=drop_cols_v6)

y_side_dev_v6 = dev_df_v6["target_side_A"].values
y_side_val_v6 = val_df_v6["target_side_A"].values

y_big_dev_v6 = dev_df_v6["target_big"].values
y_big_val_v6 = val_df_v6["target_big"].values

y_4class_val_v6 = val_df_v6["target"].map(CLASS_TO_ID).values

cat_cols_v6 = X_dev_v6.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v6:
    X_dev_v6[c] = X_dev_v6[c].fillna("Unknown").astype(str)
    X_val_v6[c] = X_val_v6[c].fillna("Unknown").astype(str)

cat_features_v6 = [X_dev_v6.columns.get_loc(c) for c in cat_cols_v6]

print("V6 Dev:", dev_df_v6.shape, dev_df_v6["Date"].min(), dev_df_v6["Date"].max())
print("V6 Val:", val_df_v6.shape, val_df_v6["Date"].min(), val_df_v6["Date"].max())
print("V6 Features:", X_dev_v6.shape[1])
print("V6 Categorical:", cat_cols_v6)
print("\nValidation target distribution:")
print(val_df_v6["target"].value_counts())

V6 Dev: (1076, 392) 2008-04-18 00:00:00 2024-05-26 00:00:00
V6 Val: (48, 392) 2025-03-22 00:00:00 2025-05-01 00:00:00
V6 Features: 387
V6 Categorical: ['team_a', 'team_b', 'venue', 'city']

Validation target distribution:
target
B_big      19
A_small    11
A_big      10
B_small     8
Name: count, dtype: int64


In [53]:
# ============================================================
# VERSION 6: TRAIN BINARY SIDE + MARGIN MODELS
# ============================================================

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

def binary_prob_positive(model, X):
    p = model.predict_proba(X)
    # CatBoost returns columns in class order [0, 1]
    return np.asarray(p)[:, 1]

def combine_side_big_probs(p_A, p_big):
    p_A = np.clip(np.asarray(p_A), 1e-6, 1 - 1e-6)
    p_big = np.clip(np.asarray(p_big), 1e-6, 1 - 1e-6)

    p_B = 1 - p_A
    p_small = 1 - p_big

    out = np.vstack([
        p_A * p_small,   # A_small
        p_A * p_big,     # A_big
        p_B * p_small,   # B_small
        p_B * p_big      # B_big
    ]).T

    return normalize_probs(out)

v6_binary_configs = [
    {
        "name": "bin_depth4",
        "depth": 4,
        "learning_rate": 0.018,
        "l2_leaf_reg": 8,
        "random_strength": 1.5,
        "bagging_temperature": 0.7,
        "iterations": 1000,
        "seed": 42
    },
    {
        "name": "bin_depth3",
        "depth": 3,
        "learning_rate": 0.025,
        "l2_leaf_reg": 10,
        "random_strength": 2.0,
        "bagging_temperature": 0.8,
        "iterations": 900,
        "seed": 42
    },
    {
        "name": "bin_depth2",
        "depth": 2,
        "learning_rate": 0.035,
        "l2_leaf_reg": 12,
        "random_strength": 2.0,
        "bagging_temperature": 0.9,
        "iterations": 800,
        "seed": 42
    }
]

v6_side_models = {}
v6_big_models = {}
v6_two_stage_preds = {}
v6_raw_results = []

for cfg in v6_binary_configs:
    print("\nTraining V6 side model:", cfg["name"])

    side_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"],
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    side_model.fit(
        X_dev_v6,
        y_side_dev_v6,
        cat_features=cat_features_v6,
        eval_set=(X_val_v6, y_side_val_v6),
        use_best_model=True
    )

    print("\nTraining V6 big/small model:", cfg["name"])

    big_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"] + 100,
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    big_model.fit(
        X_dev_v6,
        y_big_dev_v6,
        cat_features=cat_features_v6,
        eval_set=(X_val_v6, y_big_val_v6),
        use_best_model=True
    )

    p_A = binary_prob_positive(side_model, X_val_v6)
    p_big = binary_prob_positive(big_model, X_val_v6)

    p_4 = combine_side_big_probs(p_A, p_big)
    loss_4 = log_loss(y_4class_val_v6, p_4, labels=[0, 1, 2, 3])

    name = cfg["name"]
    v6_side_models[name] = side_model
    v6_big_models[name] = big_model
    v6_two_stage_preds[name] = p_4

    res = {
        "name": name,
        "raw_4class_loss": loss_4,
        "side_best_iter": side_model.best_iteration_,
        "big_best_iter": big_model.best_iteration_
    }

    v6_raw_results.append(res)
    print("\nV6 result:", res)

print("\nAll V6 two-stage raw results:")
for r in v6_raw_results:
    print(r)


Training V6 side model: bin_depth4
0:	learn: 0.6924627	test: 0.6931223	best: 0.6931223 (0)	total: 33.7ms	remaining: 33.7s
100:	learn: 0.6518681	test: 0.6888646	best: 0.6870029 (82)	total: 1.92s	remaining: 17.1s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6870029175
bestIteration = 82

Shrink model to first 83 iterations.

Training V6 big/small model: bin_depth4
0:	learn: 0.6922592	test: 0.6925391	best: 0.6925391 (0)	total: 13ms	remaining: 13s
100:	learn: 0.6377438	test: 0.6683548	best: 0.6654525 (91)	total: 1.34s	remaining: 12s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6654524529
bestIteration = 91

Shrink model to first 92 iterations.

V6 result: {'name': 'bin_depth4', 'raw_4class_loss': 1.3524553704098217, 'side_best_iter': 82, 'big_best_iter': 91}

Training V6 side model: bin_depth3
0:	learn: 0.6924359	test: 0.6929334	best: 0.6929334 (0)	total: 9.75ms	remaining: 8.77s
100:	learn: 0.6619206	test: 0.6842885	best: 0.6836297 (93)	total:

In [54]:
# ============================================================
# VERSION 6: BLEND TWO-STAGE WITH BEST V5 MULTICLASS
# ============================================================

# Rebuild best V5 validation probability from stored V5 val preds
p_v5_best_val = np.zeros_like(list(v5_val_raw_preds.values())[0])

for name, w in best_v5_public["weights"].items():
    p_v5_best_val += w * v5_val_raw_preds[name]

p_v5_best_val = p_v5_best_val / sum(best_v5_public["weights"].values())
p_v5_best_val = apply_gamma(p_v5_best_val, best_v5_public["gamma"])

# best_v5_public alpha was 1.0 in your result, but keep general:
p_v5_best_val = blend_prior(
    p_v5_best_val,
    prior_dev_v5,
    best_v5_public["alpha"]
)

p_v5_best_val = normalize_probs(p_v5_best_val)

v5_check_loss = log_loss(y_4class_val_v6, p_v5_best_val, labels=[0, 1, 2, 3])
print("Rechecked V5 validation loss:", v5_check_loss)

prior_dev_v6_4class = np.bincount(y_4class_val_v6, minlength=4)
prior_dev_v6_4class = np.bincount(dev_df_v6["target"].map(CLASS_TO_ID).values, minlength=4) / len(dev_df_v6)

best_v6_public = {
    "loss": 999,
    "two_stage_model": None,
    "blend_with_v5": None,
    "gamma": None,
    "alpha": None
}

for model_name, p_v6_raw in v6_two_stage_preds.items():
    p_v6_raw = normalize_probs(p_v6_raw)

    for blend_with_v5 in np.linspace(0.0, 1.0, 21):
        # blend_with_v5 = 1.0 means pure V5
        # blend_with_v5 = 0.0 means pure V6 two-stage
        p_mix = blend_with_v5 * p_v5_best_val + (1 - blend_with_v5) * p_v6_raw
        p_mix = normalize_probs(p_mix)

        for gamma in np.linspace(0.80, 2.20, 71):
            pg = apply_gamma(p_mix, gamma)

            for alpha in np.linspace(0.70, 1.00, 31):
                pf = blend_prior(pg, prior_dev_v6_4class, alpha)
                loss = log_loss(y_4class_val_v6, pf, labels=[0, 1, 2, 3])

                if loss < best_v6_public["loss"]:
                    best_v6_public = {
                        "loss": loss,
                        "two_stage_model": model_name,
                        "blend_with_v5": blend_with_v5,
                        "gamma": gamma,
                        "alpha": alpha
                    }

print("BEST V6 PUBLIC VALIDATION RESULT:")
print(best_v6_public)

print("\nCompare:")
print("V3 public best:", best_v3_public["loss"])
print("V4 public best:", best_v4_public["loss"])
print("V5 public best:", best_v5_public["loss"])
print("V6 public best:", best_v6_public["loss"])

Rechecked V5 validation loss: 1.3172160760603955
BEST V6 PUBLIC VALIDATION RESULT:
{'loss': 1.2645035971809668, 'two_stage_model': 'bin_depth2', 'blend_with_v5': np.float64(0.1), 'gamma': np.float64(2.2), 'alpha': np.float64(1.0)}

Compare:
V3 public best: 1.3190314025499443
V4 public best: 1.3311614645181786
V5 public best: 1.3172160760603955
V6 public best: 1.2645035971809668


In [55]:
# ============================================================
# VERSION 6: FINAL BINARY MODELS ON ALL PUBLIC TRAINING DATA
# Run this ONLY if V6 beats V5.
# ============================================================

X_all_v6_public = train_df_v6_public.drop(columns=drop_cols_v6)
y_side_all_v6 = train_df_v6_public["target_side_A"].values
y_big_all_v6 = train_df_v6_public["target_big"].values

cat_cols_v6_final = X_all_v6_public.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v6_final:
    X_all_v6_public[c] = X_all_v6_public[c].fillna("Unknown").astype(str)

cat_features_v6_final = [X_all_v6_public.columns.get_loc(c) for c in cat_cols_v6_final]

chosen_v6_name = best_v6_public["two_stage_model"]
chosen_v6_cfg = None

for cfg in v6_binary_configs:
    if cfg["name"] == chosen_v6_name:
        chosen_v6_cfg = cfg
        break

print("Chosen V6 config:", chosen_v6_cfg)

old_side_model = v6_side_models[chosen_v6_name]
old_big_model = v6_big_models[chosen_v6_name]

side_iters_final = old_side_model.best_iteration_ + 1 if old_side_model.best_iteration_ is not None else 150
big_iters_final = old_big_model.best_iteration_ + 1 if old_big_model.best_iteration_ is not None else 150

print("Final side iterations:", side_iters_final)
print("Final big iterations:", big_iters_final)

final_side_model_v6 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=side_iters_final,
    depth=chosen_v6_cfg["depth"],
    learning_rate=chosen_v6_cfg["learning_rate"],
    l2_leaf_reg=chosen_v6_cfg["l2_leaf_reg"],
    random_strength=chosen_v6_cfg["random_strength"],
    bagging_temperature=chosen_v6_cfg["bagging_temperature"],
    border_count=128,
    random_seed=chosen_v6_cfg["seed"],
    verbose=100
)

final_big_model_v6 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=big_iters_final,
    depth=chosen_v6_cfg["depth"],
    learning_rate=chosen_v6_cfg["learning_rate"],
    l2_leaf_reg=chosen_v6_cfg["l2_leaf_reg"],
    random_strength=chosen_v6_cfg["random_strength"],
    bagging_temperature=chosen_v6_cfg["bagging_temperature"],
    border_count=128,
    random_seed=chosen_v6_cfg["seed"] + 100,
    verbose=100
)

final_side_model_v6.fit(
    X_all_v6_public,
    y_side_all_v6,
    cat_features=cat_features_v6_final
)

final_big_model_v6.fit(
    X_all_v6_public,
    y_big_all_v6,
    cat_features=cat_features_v6_final
)

prior_all_v6_public = np.bincount(
    train_df_v6_public["target"].map(CLASS_TO_ID).values,
    minlength=4
) / len(train_df_v6_public)

print("Final V6 public prior:")
print(dict(zip(CLASS_ORDER, prior_all_v6_public)))

Chosen V6 config: {'name': 'bin_depth2', 'depth': 2, 'learning_rate': 0.035, 'l2_leaf_reg': 12, 'random_strength': 2.0, 'bagging_temperature': 0.9, 'iterations': 800, 'seed': 42}
Final side iterations: 480
Final big iterations: 38
0:	learn: 0.6925975	total: 7.24ms	remaining: 3.47s
100:	learn: 0.6677800	total: 578ms	remaining: 2.17s
200:	learn: 0.6463501	total: 1.14s	remaining: 1.59s
300:	learn: 0.6088668	total: 1.72s	remaining: 1.02s
400:	learn: 0.5704218	total: 2.31s	remaining: 454ms
479:	learn: 0.5447960	total: 2.74s	remaining: 0us
0:	learn: 0.6915904	total: 6.17ms	remaining: 229ms
37:	learn: 0.6683654	total: 222ms	remaining: 0us
Final V6 public prior:
{'A_small': np.float64(0.21352313167259787), 'A_big': np.float64(0.24288256227758007), 'B_small': np.float64(0.18683274021352314), 'B_big': np.float64(0.3567615658362989)}


In [56]:
# ============================================================
# VERSION 6: PUBLIC PREDICTION FUNCTIONS
# ============================================================

def predict_two_stage_public_v6_raw(feature_df):
    feature_df = feature_df.copy()

    for c in cat_cols_v6_final:
        if c not in feature_df.columns:
            feature_df[c] = "Unknown"
        feature_df[c] = feature_df[c].fillna("Unknown").astype(str)

    for c in X_all_v6_public.columns:
        if c not in feature_df.columns:
            feature_df[c] = np.nan

    feature_df = feature_df[X_all_v6_public.columns]

    p_A = binary_prob_positive(final_side_model_v6, feature_df)
    p_big = binary_prob_positive(final_big_model_v6, feature_df)

    return combine_side_big_probs(p_A, p_big)

def predict_known_toss_public_v6(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(toss_winner == team_a),
        toss_decision_bat=int(toss_decision == "bat")
    )

    feature_df = pd.DataFrame([feat])

    # V5 prediction
    p_v5 = predict_raw_public_v5(feature_df)
    p_v5 = postprocess_public_v5(p_v5)

    # V6 two-stage prediction
    p_v6 = predict_two_stage_public_v6_raw(feature_df)

    # Blend V5 + V6 using validation-selected ratio
    blend_with_v5 = best_v6_public["blend_with_v5"]
    p_mix = blend_with_v5 * p_v5 + (1 - blend_with_v5) * p_v6
    p_mix = normalize_probs(p_mix)

    p_mix = apply_gamma(p_mix, best_v6_public["gamma"])
    p_mix = blend_prior(p_mix, prior_all_v6_public, best_v6_public["alpha"])

    return normalize_probs(p_mix)[0]

In [57]:
# ============================================================
# CREATE submission_v6.csv AGAIN
# Public rows = V6 blended public model
# Private rows = V2 symmetric pre-toss model
# ============================================================

import numpy as np
import pandas as pd
import os

pred_dict_v6 = {}

# Public rows: V6 known-toss model
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss_public_v6(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict_v6[match_id] = p

# Private rows: V2 pre-toss marginalization
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict_v6[match_id] = p

submission_v6 = sample.copy()
submission_v6["match_id"] = submission_v6["match_id"].astype(str)

CLASS_ORDER = ["A_small", "A_big", "B_small", "B_big"]

for i, match_id in enumerate(submission_v6["match_id"]):
    if match_id not in pred_dict_v6:
        print("Missing prediction for:", match_id, "using V6 prior fallback")
        p = prior_all_v6_public.copy()
    else:
        p = pred_dict_v6[match_id]

    p = normalize_probs(np.asarray(p).reshape(1, -1))[0]

    submission_v6.loc[i, "A_small"] = p[0]
    submission_v6.loc[i, "A_big"] = p[1]
    submission_v6.loc[i, "B_small"] = p[2]
    submission_v6.loc[i, "B_big"] = p[3]

# Checks
assert submission_v6.shape == (53, 5), f"Wrong shape: {submission_v6.shape}"
assert submission_v6["match_id"].astype(str).tolist() == sample["match_id"].astype(str).tolist()
assert np.all(submission_v6[CLASS_ORDER].values >= 0)
assert np.all(submission_v6[CLASS_ORDER].values <= 1)
assert np.allclose(submission_v6[CLASS_ORDER].sum(axis=1).values, 1.0, atol=1e-6)

submission_v6.to_csv("submission_v6.csv", index=False)

print("✅ submission_v6.csv saved successfully")
print("Rows:", submission_v6.shape)
print("Probability row sum min:", submission_v6[CLASS_ORDER].sum(axis=1).min())
print("Probability row sum max:", submission_v6[CLASS_ORDER].sum(axis=1).max())

print("\nAverage probabilities:")
print(submission_v6[CLASS_ORDER].mean())

display(submission_v6.head())
display(submission_v6.tail())

✅ submission_v6.csv saved successfully
Rows: (53, 5)
Probability row sum min: 0.9999999999999998
Probability row sum max: 1.0000000000000002

Average probabilities:
A_small    0.130516
A_big      0.248799
B_small    0.188986
B_big      0.431698
dtype: float64


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.033419,0.074206,0.253587,0.638787
1,1473489,0.125317,0.246593,0.185175,0.442916
2,1473490,0.213445,0.447073,0.089540,0.249942
3,1473491,0.157668,0.315559,0.148263,0.378509
4,1473492,0.076921,0.152181,0.222391,0.548507


,match_id,A_small,A_big,B_small,B_big
48,M_2026_T01,0.195267,0.293354,0.203494,0.307885
49,M_2026_T02,0.198750,0.297648,0.198525,0.305077
50,M_2026_T03,0.196740,0.300049,0.200914,0.302297
51,M_2026_T04,0.204248,0.308299,0.193001,0.294452
52,M_2026_T05,0.202115,0.301531,0.197217,0.299138


Yes. This is the final submission audit stage. We should not change the model now; we only verify file format, probability safety, leakage safety, and choose the best CSV.

Thought for a couple of seconds

Yes, Prateek. Be serious now: do not keep changing the model after V6. V6 is clearly your best:

V2: 1.3651
V3: 1.3190
V4: 1.3311
V5: 1.3172
V6: 1.2645 ✅

Now run this final audit cell before Kaggle submission.

# FINAL CHECK CELL — Run before submitting

We can try one final advanced version, but we’ll keep it leakage-safe. V7 will split the margin model by winner side: A-win margin model and B-win margin model, instead of one generic big/small model.

Thought for a few seconds

Yes. Let’s try V7 to beat V6.

Your V6:

V6 best = 1.264503

To go below 1.25, the next serious idea is:

Stage 1: Predict winner side → A win or B win
Stage 2A: If A wins, predict A_small vs A_big
Stage 2B: If B wins, predict B_small vs B_big

This is better than V6 because V6 used one generic big/small model for both A and B. But margin behavior can be different:

A_big pattern ≠ B_big pattern

So V7 uses:

P(A_small) = P(A wins) × P(A small | A wins)
P(A_big)   = P(A wins) × P(A big | A wins)
P(B_small) = P(B wins) × P(B small | B wins)
P(B_big)   = P(B wins) × P(B big | B wins)

This is still leakage-safe.

V7 CELL 1 — Build conditional labels

In [58]:
# ============================================================
# VERSION 7: CONDITIONAL TWO-STAGE MODEL
# Stage 1: A/B winner
# Stage 2A: A_small vs A_big only on A-win matches
# Stage 2B: B_small vs B_big only on B-win matches
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
import numpy as np
import pandas as pd

train_df_v7_public = train_df_v3_public.copy()

def is_A_win(label):
    return 1 if str(label).startswith("A_") else 0

def is_A_big(label):
    return 1 if str(label) == "A_big" else 0

def is_B_big(label):
    return 1 if str(label) == "B_big" else 0

train_df_v7_public["target_side_A"] = train_df_v7_public["target"].apply(is_A_win)

# Conditional labels
train_df_v7_public["target_A_big"] = train_df_v7_public["target"].apply(is_A_big)
train_df_v7_public["target_B_big"] = train_df_v7_public["target"].apply(is_B_big)

print("Full target distribution:")
print(train_df_v7_public["target"].value_counts())

print("\nSide distribution:")
print(train_df_v7_public["target_side_A"].value_counts(normalize=True))

print("\nA-win matches:")
print(train_df_v7_public[train_df_v7_public["target_side_A"] == 1]["target"].value_counts())

print("\nB-win matches:")
print(train_df_v7_public[train_df_v7_public["target_side_A"] == 0]["target"].value_counts())

Full target distribution:
target
B_big      401
A_big      273
A_small    240
B_small    210
Name: count, dtype: int64

Side distribution:
target_side_A
0    0.543594
1    0.456406
Name: proportion, dtype: float64

A-win matches:
target
A_big      273
A_small    240
Name: count, dtype: int64

B-win matches:
target
B_big      401
B_small    210
Name: count, dtype: int64


In [59]:
# ============================================================
# VERSION 7: TIME VALIDATION SPLIT
# ============================================================

drop_cols_v7 = [
    "target",
    "target_side_A",
    "target_A_big",
    "target_B_big",
    "Date",
    "match_id"
]

val_mask_v7 = train_df_v7_public["Date"] >= pd.Timestamp("2025-01-01")

if val_mask_v7.sum() < 30:
    cutoff_idx = int(len(train_df_v7_public) * 0.85)
    cutoff_date = train_df_v7_public["Date"].iloc[cutoff_idx]
    val_mask_v7 = train_df_v7_public["Date"] >= cutoff_date

dev_df_v7 = train_df_v7_public[~val_mask_v7].copy()
val_df_v7 = train_df_v7_public[val_mask_v7].copy()

X_dev_v7 = dev_df_v7.drop(columns=drop_cols_v7)
X_val_v7 = val_df_v7.drop(columns=drop_cols_v7)

y_side_dev_v7 = dev_df_v7["target_side_A"].values
y_side_val_v7 = val_df_v7["target_side_A"].values

y_4class_val_v7 = val_df_v7["target"].map(CLASS_TO_ID).values

cat_cols_v7 = X_dev_v7.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols_v7:
    X_dev_v7[c] = X_dev_v7[c].fillna("Unknown").astype(str)
    X_val_v7[c] = X_val_v7[c].fillna("Unknown").astype(str)

cat_features_v7 = [X_dev_v7.columns.get_loc(c) for c in cat_cols_v7]

print("V7 Dev:", dev_df_v7.shape, dev_df_v7["Date"].min(), dev_df_v7["Date"].max())
print("V7 Val:", val_df_v7.shape, val_df_v7["Date"].min(), val_df_v7["Date"].max())
print("V7 Features:", X_dev_v7.shape[1])
print("V7 categorical:", cat_cols_v7)

print("\nV7 validation distribution:")
print(val_df_v7["target"].value_counts())

V7 Dev: (1076, 393) 2008-04-18 00:00:00 2024-05-26 00:00:00
V7 Val: (48, 393) 2025-03-22 00:00:00 2025-05-01 00:00:00
V7 Features: 387
V7 categorical: ['team_a', 'team_b', 'venue', 'city']

V7 validation distribution:
target
B_big      19
A_small    11
A_big      10
B_small     8
Name: count, dtype: int64


In [60]:
# ============================================================
# VERSION 7: CONDITIONAL DATASETS
# ============================================================

# A-win subset for A_small vs A_big
dev_A_v7 = dev_df_v7[dev_df_v7["target_side_A"] == 1].copy()
val_A_v7 = val_df_v7[val_df_v7["target_side_A"] == 1].copy()

X_dev_A_v7 = dev_A_v7.drop(columns=drop_cols_v7)
y_dev_A_big_v7 = dev_A_v7["target_A_big"].values

X_val_A_v7 = val_A_v7.drop(columns=drop_cols_v7)
y_val_A_big_v7 = val_A_v7["target_A_big"].values

# B-win subset for B_small vs B_big
dev_B_v7 = dev_df_v7[dev_df_v7["target_side_A"] == 0].copy()
val_B_v7 = val_df_v7[val_df_v7["target_side_A"] == 0].copy()

X_dev_B_v7 = dev_B_v7.drop(columns=drop_cols_v7)
y_dev_B_big_v7 = dev_B_v7["target_B_big"].values

X_val_B_v7 = val_B_v7.drop(columns=drop_cols_v7)
y_val_B_big_v7 = val_B_v7["target_B_big"].values

for df in [X_dev_A_v7, X_val_A_v7, X_dev_B_v7, X_val_B_v7]:
    for c in cat_cols_v7:
        df[c] = df[c].fillna("Unknown").astype(str)

print("A conditional dev:", X_dev_A_v7.shape, "val:", X_val_A_v7.shape)
print("A conditional target:", pd.Series(y_dev_A_big_v7).value_counts(normalize=True))

print("\nB conditional dev:", X_dev_B_v7.shape, "val:", X_val_B_v7.shape)
print("B conditional target:", pd.Series(y_dev_B_big_v7).value_counts(normalize=True))

A conditional dev: (492, 387) val: (21, 387)
A conditional target: 1    0.534553
0    0.465447
Name: proportion, dtype: float64

B conditional dev: (584, 387) val: (27, 387)
B conditional target: 1    0.65411
0    0.34589
Name: proportion, dtype: float64


In [61]:
# ============================================================
# VERSION 7: HELPERS
# ============================================================

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-8, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def apply_gamma(p, gamma):
    p = normalize_probs(p)
    p = p ** gamma
    return normalize_probs(p)

def blend_prior(p, prior, alpha):
    p = normalize_probs(p)
    prior = np.asarray(prior).reshape(1, -1)
    return normalize_probs(alpha * p + (1 - alpha) * prior)

def binary_prob_positive(model, X):
    p = model.predict_proba(X)
    return np.asarray(p)[:, 1]

def combine_conditional_probs(p_A_win, p_A_big_given_A, p_B_big_given_B):
    p_A_win = np.clip(np.asarray(p_A_win), 1e-6, 1 - 1e-6)
    p_B_win = 1 - p_A_win

    p_A_big_given_A = np.clip(np.asarray(p_A_big_given_A), 1e-6, 1 - 1e-6)
    p_B_big_given_B = np.clip(np.asarray(p_B_big_given_B), 1e-6, 1 - 1e-6)

    p_A_small_given_A = 1 - p_A_big_given_A
    p_B_small_given_B = 1 - p_B_big_given_B

    out = np.vstack([
        p_A_win * p_A_small_given_A,
        p_A_win * p_A_big_given_A,
        p_B_win * p_B_small_given_B,
        p_B_win * p_B_big_given_B
    ]).T

    return normalize_probs(out)

In [62]:
# ============================================================
# VERSION 7: TRAIN SIDE + CONDITIONAL MARGIN MODELS
# ============================================================

v7_configs = [
    {
        "name": "v7_depth2",
        "depth": 2,
        "learning_rate": 0.035,
        "l2_leaf_reg": 12,
        "random_strength": 2.0,
        "bagging_temperature": 0.9,
        "iterations": 900,
        "seed": 42
    },
    {
        "name": "v7_depth2_reg",
        "depth": 2,
        "learning_rate": 0.030,
        "l2_leaf_reg": 18,
        "random_strength": 2.5,
        "bagging_temperature": 1.0,
        "iterations": 1000,
        "seed": 99
    },
    {
        "name": "v7_depth3",
        "depth": 3,
        "learning_rate": 0.025,
        "l2_leaf_reg": 10,
        "random_strength": 2.0,
        "bagging_temperature": 0.8,
        "iterations": 900,
        "seed": 42
    }
]

v7_side_models = {}
v7_A_margin_models = {}
v7_B_margin_models = {}
v7_preds = {}
v7_results = []

for cfg in v7_configs:
    print("\nTraining config:", cfg["name"])

    # Winner side model
    side_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"],
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    side_model.fit(
        X_dev_v7,
        y_side_dev_v7,
        cat_features=cat_features_v7,
        eval_set=(X_val_v7, y_side_val_v7),
        use_best_model=True
    )

    # A margin model: A_small vs A_big
    A_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"] + 11,
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    A_model.fit(
        X_dev_A_v7,
        y_dev_A_big_v7,
        cat_features=cat_features_v7,
        eval_set=(X_val_A_v7, y_val_A_big_v7),
        use_best_model=True
    )

    # B margin model: B_small vs B_big
    B_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=cfg["iterations"],
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        border_count=128,
        random_seed=cfg["seed"] + 22,
        od_type="Iter",
        od_wait=80,
        verbose=100
    )

    B_model.fit(
        X_dev_B_v7,
        y_dev_B_big_v7,
        cat_features=cat_features_v7,
        eval_set=(X_val_B_v7, y_val_B_big_v7),
        use_best_model=True
    )

    p_A_win = binary_prob_positive(side_model, X_val_v7)
    p_A_big_given_A = binary_prob_positive(A_model, X_val_v7)
    p_B_big_given_B = binary_prob_positive(B_model, X_val_v7)

    p_4 = combine_conditional_probs(
        p_A_win=p_A_win,
        p_A_big_given_A=p_A_big_given_A,
        p_B_big_given_B=p_B_big_given_B
    )

    raw_loss = log_loss(y_4class_val_v7, p_4, labels=[0, 1, 2, 3])

    name = cfg["name"]

    v7_side_models[name] = side_model
    v7_A_margin_models[name] = A_model
    v7_B_margin_models[name] = B_model
    v7_preds[name] = p_4

    res = {
        "name": name,
        "raw_loss": raw_loss,
        "side_iter": side_model.best_iteration_,
        "A_margin_iter": A_model.best_iteration_,
        "B_margin_iter": B_model.best_iteration_
    }

    v7_results.append(res)
    print("\nV7 result:", res)

print("\nAll V7 raw results:")
for r in v7_results:
    print(r)


Training config: v7_depth2
0:	learn: 0.6925826	test: 0.6929204	best: 0.6929204 (0)	total: 6.92ms	remaining: 6.22s
100:	learn: 0.6683891	test: 0.6817278	best: 0.6789738 (74)	total: 601ms	remaining: 4.75s
200:	learn: 0.6461625	test: 0.6708088	best: 0.6698441 (190)	total: 1.18s	remaining: 4.09s
300:	learn: 0.6064226	test: 0.6422348	best: 0.6409575 (298)	total: 1.75s	remaining: 3.49s
400:	learn: 0.5691660	test: 0.6355019	best: 0.6355019 (400)	total: 2.3s	remaining: 2.87s
500:	learn: 0.5361833	test: 0.6355490	best: 0.6303850 (479)	total: 2.88s	remaining: 2.29s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6303849535
bestIteration = 479

Shrink model to first 480 iterations.
0:	learn: 0.6926568	test: 0.6932437	best: 0.6932437 (0)	total: 5.34ms	remaining: 4.8s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6932437267
bestIteration = 0

Shrink model to first 1 iterations.
0:	learn: 0.6897146	test: 0.6889366	best: 0.6889366 (0)	total: 5.95ms	remaining

In [63]:
# ============================================================
# VERSION 7 FAST BLEND SEARCH
# Replaces the slow interrupted V7 blend cell
# ============================================================

from sklearn.metrics import log_loss
import numpy as np

# Rebuild V5 validation probabilities
p_v5_val = np.zeros_like(list(v5_val_raw_preds.values())[0])

for name, w in best_v5_public["weights"].items():
    p_v5_val += w * v5_val_raw_preds[name]

p_v5_val = p_v5_val / sum(best_v5_public["weights"].values())
p_v5_val = apply_gamma(p_v5_val, best_v5_public["gamma"])
p_v5_val = blend_prior(p_v5_val, prior_dev_v5, best_v5_public["alpha"])
p_v5_val = normalize_probs(p_v5_val)

# Rebuild V6 validation probabilities
p_v6_raw_best = v6_two_stage_preds[best_v6_public["two_stage_model"]]

p_v6_mix = (
    best_v6_public["blend_with_v5"] * p_v5_val
    + (1 - best_v6_public["blend_with_v5"]) * p_v6_raw_best
)

p_v6_mix = normalize_probs(p_v6_mix)
p_v6_mix = apply_gamma(p_v6_mix, best_v6_public["gamma"])

prior_dev_v6_4class = np.bincount(
    dev_df_v6["target"].map(CLASS_TO_ID).values,
    minlength=4
) / len(dev_df_v6)

p_v6_val = blend_prior(
    p_v6_mix,
    prior_dev_v6_4class,
    best_v6_public["alpha"]
)

p_v6_val = normalize_probs(p_v6_val)

print("V5 check:", log_loss(y_4class_val_v7, p_v5_val, labels=[0, 1, 2, 3]))
print("V6 check:", log_loss(y_4class_val_v7, p_v6_val, labels=[0, 1, 2, 3]))

prior_dev_v7_4class = np.bincount(
    dev_df_v7["target"].map(CLASS_TO_ID).values,
    minlength=4
) / len(dev_df_v7)

best_v7_public = {
    "loss": 999,
    "v7_model": None,
    "w_v5": None,
    "w_v6": None,
    "w_v7": None,
    "gamma": None,
    "alpha": None
}

# Much smaller, smarter grid
weight_sets = [
    # mostly V6, small V7
    (0.0, 0.90, 0.10),
    (0.0, 0.80, 0.20),
    (0.0, 0.70, 0.30),

    # V6 + V5 + V7
    (0.10, 0.80, 0.10),
    (0.10, 0.70, 0.20),
    (0.10, 0.60, 0.30),

    # mostly V7
    (0.0, 0.40, 0.60),
    (0.0, 0.30, 0.70),
    (0.0, 0.20, 0.80),

    # pure checks
    (0.0, 1.00, 0.0),
    (0.0, 0.0, 1.0),
]

gamma_grid = np.linspace(1.6, 2.6, 21)
alpha_grid = [1.0, 0.95, 0.90]

for v7_name, p_v7_raw in v7_preds.items():
    p_v7_raw = normalize_probs(p_v7_raw)

    for w_v5, w_v6, w_v7 in weight_sets:
        total = w_v5 + w_v6 + w_v7

        p_mix = (
            w_v5 * p_v5_val +
            w_v6 * p_v6_val +
            w_v7 * p_v7_raw
        ) / total

        p_mix = normalize_probs(p_mix)

        for gamma in gamma_grid:
            pg = apply_gamma(p_mix, gamma)

            for alpha in alpha_grid:
                pf = blend_prior(pg, prior_dev_v7_4class, alpha)
                loss = log_loss(y_4class_val_v7, pf, labels=[0, 1, 2, 3])

                if loss < best_v7_public["loss"]:
                    best_v7_public = {
                        "loss": loss,
                        "v7_model": v7_name,
                        "w_v5": w_v5,
                        "w_v6": w_v6,
                        "w_v7": w_v7,
                        "gamma": gamma,
                        "alpha": alpha
                    }

print("BEST V7 FAST PUBLIC VALIDATION RESULT:")
print(best_v7_public)

print("\nCompare:")
print("V5:", best_v5_public["loss"])
print("V6:", best_v6_public["loss"])
print("V7 fast:", best_v7_public["loss"])

V5 check: 1.3172160760603955
V6 check: 1.2645035971809666
BEST V7 FAST PUBLIC VALIDATION RESULT:
{'loss': 1.229471590006692, 'v7_model': 'v7_depth2', 'w_v5': 0.0, 'w_v6': 0.0, 'w_v7': 1.0, 'gamma': np.float64(2.1500000000000004), 'alpha': 1.0}

Compare:
V5: 1.3172160760603955
V6: 1.2645035971809668
V7 fast: 1.229471590006692


In [64]:
# ============================================================
# VERSION 7: FINAL TRAINING ON ALL PUBLIC DATA
# Run only if V7 beats V6.
# ============================================================

X_all_v7_public = train_df_v7_public.drop(columns=drop_cols_v7)

y_side_all_v7 = train_df_v7_public["target_side_A"].values

A_all_df_v7 = train_df_v7_public[train_df_v7_public["target_side_A"] == 1].copy()
B_all_df_v7 = train_df_v7_public[train_df_v7_public["target_side_A"] == 0].copy()

X_A_all_v7 = A_all_df_v7.drop(columns=drop_cols_v7)
y_A_big_all_v7 = A_all_df_v7["target_A_big"].values

X_B_all_v7 = B_all_df_v7.drop(columns=drop_cols_v7)
y_B_big_all_v7 = B_all_df_v7["target_B_big"].values

cat_cols_v7_final = X_all_v7_public.select_dtypes(include=["object"]).columns.tolist()

for df in [X_all_v7_public, X_A_all_v7, X_B_all_v7]:
    for c in cat_cols_v7_final:
        df[c] = df[c].fillna("Unknown").astype(str)

cat_features_v7_final = [X_all_v7_public.columns.get_loc(c) for c in cat_cols_v7_final]

chosen_v7_name = best_v7_public["v7_model"]
chosen_v7_cfg = None

for cfg in v7_configs:
    if cfg["name"] == chosen_v7_name:
        chosen_v7_cfg = cfg
        break

print("Chosen V7 config:", chosen_v7_cfg)

old_side = v7_side_models[chosen_v7_name]
old_A = v7_A_margin_models[chosen_v7_name]
old_B = v7_B_margin_models[chosen_v7_name]

side_iters = old_side.best_iteration_ + 1 if old_side.best_iteration_ is not None else 150
A_iters = old_A.best_iteration_ + 1 if old_A.best_iteration_ is not None else 100
B_iters = old_B.best_iteration_ + 1 if old_B.best_iteration_ is not None else 100

print("Final iters:", side_iters, A_iters, B_iters)

final_side_model_v7 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=side_iters,
    depth=chosen_v7_cfg["depth"],
    learning_rate=chosen_v7_cfg["learning_rate"],
    l2_leaf_reg=chosen_v7_cfg["l2_leaf_reg"],
    random_strength=chosen_v7_cfg["random_strength"],
    bagging_temperature=chosen_v7_cfg["bagging_temperature"],
    border_count=128,
    random_seed=chosen_v7_cfg["seed"],
    verbose=100
)

final_A_margin_model_v7 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=A_iters,
    depth=chosen_v7_cfg["depth"],
    learning_rate=chosen_v7_cfg["learning_rate"],
    l2_leaf_reg=chosen_v7_cfg["l2_leaf_reg"],
    random_strength=chosen_v7_cfg["random_strength"],
    bagging_temperature=chosen_v7_cfg["bagging_temperature"],
    border_count=128,
    random_seed=chosen_v7_cfg["seed"] + 11,
    verbose=100
)

final_B_margin_model_v7 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=B_iters,
    depth=chosen_v7_cfg["depth"],
    learning_rate=chosen_v7_cfg["learning_rate"],
    l2_leaf_reg=chosen_v7_cfg["l2_leaf_reg"],
    random_strength=chosen_v7_cfg["random_strength"],
    bagging_temperature=chosen_v7_cfg["bagging_temperature"],
    border_count=128,
    random_seed=chosen_v7_cfg["seed"] + 22,
    verbose=100
)

final_side_model_v7.fit(
    X_all_v7_public,
    y_side_all_v7,
    cat_features=cat_features_v7_final
)

final_A_margin_model_v7.fit(
    X_A_all_v7,
    y_A_big_all_v7,
    cat_features=cat_features_v7_final
)

final_B_margin_model_v7.fit(
    X_B_all_v7,
    y_B_big_all_v7,
    cat_features=cat_features_v7_final
)

prior_all_v7_public = np.bincount(
    train_df_v7_public["target"].map(CLASS_TO_ID).values,
    minlength=4
) / len(train_df_v7_public)

print("Final V7 public prior:")
print(dict(zip(CLASS_ORDER, prior_all_v7_public)))

Chosen V7 config: {'name': 'v7_depth2', 'depth': 2, 'learning_rate': 0.035, 'l2_leaf_reg': 12, 'random_strength': 2.0, 'bagging_temperature': 0.9, 'iterations': 900, 'seed': 42}
Final iters: 480 1 90
0:	learn: 0.6925975	total: 7.4ms	remaining: 3.54s
100:	learn: 0.6677800	total: 604ms	remaining: 2.27s
200:	learn: 0.6463501	total: 1.22s	remaining: 1.69s
300:	learn: 0.6088668	total: 1.79s	remaining: 1.06s
400:	learn: 0.5704218	total: 2.38s	remaining: 469ms
479:	learn: 0.5447960	total: 2.83s	remaining: 0us
0:	learn: 0.6929742	total: 7.24ms	remaining: 0us
0:	learn: 0.6894729	total: 5.56ms	remaining: 495ms
89:	learn: 0.6105557	total: 696ms	remaining: 0us
Final V7 public prior:
{'A_small': np.float64(0.21352313167259787), 'A_big': np.float64(0.24288256227758007), 'B_small': np.float64(0.18683274021352314), 'B_big': np.float64(0.3567615658362989)}


In [65]:
# ============================================================
# VERSION 7: PUBLIC PREDICTION FUNCTIONS
# ============================================================

def predict_v7_raw_public(feature_df):
    feature_df = feature_df.copy()

    for c in cat_cols_v7_final:
        if c not in feature_df.columns:
            feature_df[c] = "Unknown"
        feature_df[c] = feature_df[c].fillna("Unknown").astype(str)

    for c in X_all_v7_public.columns:
        if c not in feature_df.columns:
            feature_df[c] = np.nan

    feature_df = feature_df[X_all_v7_public.columns]

    p_A_win = binary_prob_positive(final_side_model_v7, feature_df)
    p_A_big_given_A = binary_prob_positive(final_A_margin_model_v7, feature_df)
    p_B_big_given_B = binary_prob_positive(final_B_margin_model_v7, feature_df)

    return combine_conditional_probs(
        p_A_win=p_A_win,
        p_A_big_given_A=p_A_big_given_A,
        p_B_big_given_B=p_B_big_given_B
    )

def predict_known_toss_public_v7(date, team_a, team_b, venue, city, toss_winner, toss_decision):
    feat = make_feature_row_v2(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner_is_A=int(toss_winner == team_a),
        toss_decision_bat=int(toss_decision == "bat")
    )

    feature_df = pd.DataFrame([feat])

    # V5 probability
    p_v5 = predict_raw_public_v5(feature_df)
    p_v5 = postprocess_public_v5(p_v5)

    # V6 probability
    p_v6 = predict_known_toss_public_v6(
        date=date,
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        city=city,
        toss_winner=toss_winner,
        toss_decision=toss_decision
    ).reshape(1, -1)

    # V7 probability
    p_v7 = predict_v7_raw_public(feature_df)

    w_v5 = best_v7_public["w_v5"]
    w_v6 = best_v7_public["w_v6"]
    w_v7 = best_v7_public["w_v7"]

    total = w_v5 + w_v6 + w_v7

    p_mix = (w_v5 * p_v5 + w_v6 * p_v6 + w_v7 * p_v7) / total
    p_mix = normalize_probs(p_mix)

    p_mix = apply_gamma(p_mix, best_v7_public["gamma"])
    p_mix = blend_prior(p_mix, prior_all_v7_public, best_v7_public["alpha"])

    return normalize_probs(p_mix)[0]

In [66]:
# ============================================================
# VERSION 7: CREATE FINAL HYBRID SUBMISSION
# Public rows = V7 public model
# Private rows = V2 symmetric pre-toss model
# ============================================================

pred_dict_v7 = {}

# Public rows: V7 public known-toss model
for _, r in public_lb.iterrows():
    match_id = str(r["match_id"])

    p = predict_known_toss_public_v7(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"],
        toss_winner=r["toss_winner"],
        toss_decision=r["toss_decision"]
    )

    pred_dict_v7[match_id] = p

# Private rows: V2 pre-toss marginalization remains safer
for _, r in schedule.iterrows():
    match_id = str(r["match_id"])

    p = predict_private_pretoss_v2(
        date=r["date"],
        team_a=r["team_a"],
        team_b=r["team_b"],
        venue=r["venue"],
        city=r["city"]
    )

    pred_dict_v7[match_id] = p

submission_v7 = sample.copy()
submission_v7["match_id"] = submission_v7["match_id"].astype(str)

for i, match_id in enumerate(submission_v7["match_id"]):
    if match_id not in pred_dict_v7:
        print("Missing prediction for:", match_id, "using V7 prior fallback")
        p = prior_all_v7_public.copy()
    else:
        p = pred_dict_v7[match_id]

    p = normalize_probs(np.asarray(p).reshape(1, -1))[0]

    submission_v7.loc[i, "A_small"] = p[0]
    submission_v7.loc[i, "A_big"] = p[1]
    submission_v7.loc[i, "B_small"] = p[2]
    submission_v7.loc[i, "B_big"] = p[3]

# Hard checks
assert submission_v7.shape == (53, 5), f"Wrong shape: {submission_v7.shape}"
assert submission_v7["match_id"].astype(str).tolist() == sample["match_id"].astype(str).tolist()
assert np.all(submission_v7[CLASS_ORDER].values >= 0)
assert np.all(submission_v7[CLASS_ORDER].values <= 1)
assert np.allclose(submission_v7[CLASS_ORDER].sum(axis=1).values, 1.0, atol=1e-6)

submission_v7.to_csv("submission_v7.csv", index=False)

print("submission_v7.csv saved successfully")
print("Rows:", submission_v7.shape)
print("Probability row sum min:", submission_v7[CLASS_ORDER].sum(axis=1).min())
print("Probability row sum max:", submission_v7[CLASS_ORDER].sum(axis=1).max())

print("\nAverage probabilities:")
print(submission_v7[CLASS_ORDER].mean())

display(submission_v7.head())
display(submission_v7.tail())

submission_v7.csv saved successfully
Rows: (53, 5)
Probability row sum min: 1.0
Probability row sum max: 1.0000000000000002

Average probabilities:
A_small    0.178127
A_big      0.189241
B_small    0.129719
B_big      0.502913
dtype: float64


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.040040,0.040444,0.147114,0.772402
1,1473489,0.173279,0.175029,0.118172,0.533520
2,1473490,0.325463,0.328750,0.055955,0.289831
3,1473491,0.226265,0.228550,0.085339,0.459847
4,1473492,0.103350,0.104394,0.139185,0.653071


,match_id,A_small,A_big,B_small,B_big
48,M_2026_T01,0.195267,0.293354,0.203494,0.307885
49,M_2026_T02,0.198750,0.297648,0.198525,0.305077
50,M_2026_T03,0.196740,0.300049,0.200914,0.302297
51,M_2026_T04,0.204248,0.308299,0.193001,0.294452
52,M_2026_T05,0.202115,0.301531,0.197217,0.299138


#Summary OF ALL Versions and log loss

In [67]:
# ============================================================
# FINAL MODEL SCORE SUMMARY: V1 TO V7
# Run this after all model/version cells
# ============================================================

import numpy as np
import pandas as pd

summary_rows = []

def add_score(version, score_var=None, score_value=None, note=""):
    try:
        if score_value is not None:
            score = float(score_value)
        elif score_var is not None:
            score = float(score_var)
        else:
            score = np.nan

        summary_rows.append({
            "Version": version,
            "Validation_LogLoss": score,
            "Note": note
        })
    except Exception as e:
        summary_rows.append({
            "Version": version,
            "Validation_LogLoss": np.nan,
            "Note": f"Not available: {e}"
        })

# -----------------------------
# V1
# -----------------------------
try:
    add_score(
        "V1",
        score_value=best["loss"],
        note="Initial ensemble baseline"
    )
except Exception:
    add_score("V1", note="best['loss'] not found")

# -----------------------------
# V2
# -----------------------------
try:
    add_score(
        "V2",
        score_value=best_v2["loss"],
        note="CatBoost + stronger features"
    )
except Exception:
    add_score("V2", note="best_v2['loss'] not found")

# -----------------------------
# V3
# -----------------------------
try:
    add_score(
        "V3",
        score_value=best_v3_public["loss"],
        note="Original-only public known-toss model"
    )
except Exception:
    add_score("V3", note="best_v3_public['loss'] not found")

# -----------------------------
# V4
# -----------------------------
try:
    add_score(
        "V4",
        score_value=best_v4_public["loss"],
        note="Role-specific features"
    )
except Exception:
    add_score("V4", note="best_v4_public['loss'] not found")

# -----------------------------
# V5
# -----------------------------
try:
    add_score(
        "V5",
        score_value=best_v5_public["loss"],
        note="V3 features + recency weights + multi-seed search"
    )
except Exception:
    add_score("V5", note="best_v5_public['loss'] not found")

# -----------------------------
# V6
# -----------------------------
try:
    add_score(
        "V6",
        score_value=best_v6_public["loss"],
        note="Two-stage side + big/small model"
    )
except Exception:
    add_score("V6", note="best_v6_public['loss'] not found")

# -----------------------------
# V7
# -----------------------------
try:
    add_score(
        "V7",
        score_value=best_v7_public["loss"],
        note="Conditional two-stage model"
    )
except Exception:
    add_score("V7", note="best_v7_public['loss'] not found")

summary_df = pd.DataFrame(summary_rows)

summary_df["Validation_LogLoss"] = pd.to_numeric(
    summary_df["Validation_LogLoss"],
    errors="coerce"
)

summary_df = summary_df.sort_values(
    "Validation_LogLoss",
    na_position="last"
).reset_index(drop=True)

print("========== FINAL VALIDATION LOG LOSS SUMMARY ==========")
display(summary_df)

best_row = summary_df.dropna(subset=["Validation_LogLoss"]).iloc[0]

print("\n🏆 BEST LOCAL VALIDATION MODEL")
print("Version:", best_row["Version"])
print("Log Loss:", best_row["Validation_LogLoss"])
print("Note:", best_row["Note"])

print("\nREFERENCE")
print("Uniform baseline: ~1.3863")
print("Solid ML model: 1.20 - 1.30")
print("Strong submission: 0.95 - 1.10")

print("\nIMPORTANT")
print("Lower log loss is better.")
print("If choosing safest final submission, compare score + overconfidence risk, not score alone.")

========== FINAL VALIDATION LOG LOSS SUMMARY ==========


,Version,Validation_LogLoss,Note
0,V7,1.229472,Conditional two-stage model
1,V6,1.264504,Two-stage side + big/small model
2,V5,1.317216,V3 features + recency weights + multi-seed search
3,V3,1.319031,Original-only public known-toss model
4,V4,1.331161,Role-specific features
5,V2,1.365194,CatBoost + stronger features
6,V1,1.387600,Initial ensemble baseline



🏆 BEST LOCAL VALIDATION MODEL
Version: V7
Log Loss: 1.229471590006692
Note: Conditional two-stage model

REFERENCE
Uniform baseline: ~1.3863
Solid ML model: 1.20 - 1.30
Strong submission: 0.95 - 1.10

IMPORTANT
Lower log loss is better.
If choosing safest final submission, compare score + overconfidence risk, not score alone.


In [68]:
# ============================================================
# FINAL ACCURACY SUMMARY: V1 TO V7
# ============================================================

from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

accuracy_rows = []

def add_accuracy(version, y_true, pred_probs, note=""):
    try:
        pred_class = np.argmax(pred_probs, axis=1)
        acc = accuracy_score(y_true, pred_class)
        accuracy_rows.append({
            "Version": version,
            "Accuracy": acc,
            "Accuracy_%": acc * 100,
            "Note": note
        })
    except Exception as e:
        accuracy_rows.append({
            "Version": version,
            "Accuracy": np.nan,
            "Accuracy_%": np.nan,
            "Note": f"Not available: {e}"
        })

# V1 ensemble validation prediction
try:
    p_v1 = np.zeros_like(list(val_preds.values())[0])
    total_w = sum(best["weights"].values())
    for name, w in best["weights"].items():
        p_v1 += w * val_preds[name]
    p_v1 = p_v1 / total_w
    add_accuracy("V1", y_val, p_v1, "Initial ensemble")
except Exception as e:
    accuracy_rows.append({"Version": "V1", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

# V2
try:
    add_accuracy("V2", y_val_v2, val_raw_v2, "CatBoost V2")
except Exception as e:
    accuracy_rows.append({"Version": "V2", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

# V3
try:
    best_v3_name = list(best_v3_public["weights"].keys())[0]
    add_accuracy("V3", y_val_v3, v3_val_raw_preds[best_v3_name], "Public original-only model")
except Exception as e:
    accuracy_rows.append({"Version": "V3", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

# V4
try:
    best_v4_name = list(best_v4_public["weights"].keys())[0]
    add_accuracy("V4", y_val_v4, v4_val_raw_preds[best_v4_name], "Role feature model")
except Exception as e:
    accuracy_rows.append({"Version": "V4", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

# V5
try:
    best_v5_name = list(best_v5_public["weights"].keys())[0]
    add_accuracy("V5", y_val_v5, v5_val_raw_preds[best_v5_name], "Recency weighted model")
except Exception as e:
    accuracy_rows.append({"Version": "V5", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

# V6
try:
    p_v6 = v6_two_stage_preds[best_v6_public["two_stage_model"]]
    add_accuracy("V6", y_4class_val_v6, p_v6, "Two-stage side + margin model")
except Exception as e:
    accuracy_rows.append({"Version": "V6", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

# V7
try:
    p_v7 = v7_preds[best_v7_public["v7_model"]]
    add_accuracy("V7", y_4class_val_v7, p_v7, "Conditional two-stage model")
except Exception as e:
    accuracy_rows.append({"Version": "V7", "Accuracy": np.nan, "Accuracy_%": np.nan, "Note": str(e)})

accuracy_df = pd.DataFrame(accuracy_rows)
accuracy_df = accuracy_df.sort_values("Accuracy_%", ascending=False, na_position="last").reset_index(drop=True)

display(accuracy_df)

best_acc = accuracy_df.dropna(subset=["Accuracy_%"]).iloc[0]

print("BEST ACCURACY MODEL:")
print("Version:", best_acc["Version"])
print("Accuracy:", round(best_acc["Accuracy_%"], 2), "%")

,Version,Accuracy,Accuracy_%,Note
0,V6,0.500000,50.000000,Two-stage side + margin model
1,V7,0.458333,45.833333,Conditional two-stage model
2,V5,0.437500,43.750000,Recency weighted model
3,V3,0.395833,39.583333,Public original-only model
4,V4,0.395833,39.583333,Role feature model
5,V2,0.302083,30.208333,CatBoost V2
6,V1,0.250000,25.000000,Initial ensemble


BEST ACCURACY MODEL:
Version: V6
Accuracy: 50.0 %


In [113]:
# ============================================================
# V8 CALIBRATION LADDER FROM BEST PUBLIC CSV
# No leakage: only calibrates our own prediction probabilities
# ============================================================

import pandas as pd
import numpy as np
import os

BEST_FILE = "best_public_135961.csv"
assert os.path.exists(BEST_FILE), f"{BEST_FILE} not found. Upload/rename your best CSV first."

base = pd.read_csv(BEST_FILE)

prob_cols = ["A_small", "A_big", "B_small", "B_big"]
required_cols = ["match_id", "A_small", "A_big", "B_small", "B_big"]

assert base.shape == (53, 5)
assert base.columns.tolist() == required_cols
assert base[prob_cols].isna().sum().sum() == 0
assert np.isfinite(base[prob_cols].values).all()
assert (base[prob_cols].values >= 0).all()
assert (base[prob_cols].values <= 1).all()
assert np.allclose(base[prob_cols].sum(axis=1), 1.0, atol=1e-6)

uniform = np.array([0.25, 0.25, 0.25, 0.25])

# Softening weights
# 0.95 = tiny change, 0.80 = stronger safety
weights = [0.95, 0.92, 0.90, 0.88, 0.85, 0.82, 0.80]

for w in weights:
    out = base.copy()
    out[prob_cols] = w * base[prob_cols].values + (1 - w) * uniform
    out[prob_cols] = out[prob_cols].clip(1e-8, 1)
    out[prob_cols] = out[prob_cols].div(out[prob_cols].sum(axis=1), axis=0)

    filename = f"submission_v8_soft_{int(w*100)}.csv"
    out.to_csv(filename, index=False)

    print("\nSaved:", filename)
    print("Weight on model:", w)
    print("Average probabilities:")
    print(out[prob_cols].mean())
    print("Max probability:", out[prob_cols].max(axis=1).max())
    print("Row sum min/max:", out[prob_cols].sum(axis=1).min(), out[prob_cols].sum(axis=1).max())


Saved: submission_v8_soft_95.csv
Weight on model: 0.95
Average probabilities:
A_small    0.198404
A_big      0.238493
B_small    0.165293
B_big      0.397810
dtype: float64
Max probability: 0.44677154445507167
Row sum min/max: 1.0 1.0000000000000002

Saved: submission_v8_soft_92.csv
Weight on model: 0.92
Average probabilities:
A_small    0.200033
A_big      0.238857
B_small    0.167968
B_big      0.393142
dtype: float64
Max probability: 0.440557706209122
Row sum min/max: 1.0 1.0000000000000002

Saved: submission_v8_soft_90.csv
Weight on model: 0.9
Average probabilities:
A_small    0.201120
A_big      0.239099
B_small    0.169751
B_big      0.390030
dtype: float64
Max probability: 0.4364151473784889
Row sum min/max: 1.0 1.0000000000000002

Saved: submission_v8_soft_88.csv
Weight on model: 0.88
Average probabilities:
A_small    0.202206
A_big      0.239341
B_small    0.171534
B_big      0.386918
dtype: float64
Max probability: 0.43227258854785583
Row sum min/max: 1.0 1.0

Saved: submiss

In [114]:

# ============================================================
# FINAL KAGGLE PUSH
# ============================================================

import pandas as pd
import numpy as np

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-12, 1)
    return p / p.sum(axis=1, keepdims=True)

# LOAD BEST FILE
sub = pd.read_csv("submission_v8_soft_80.csv")

cols = ["A_small", "A_big", "B_small", "B_big"]

p = sub[cols].values.copy()

# ------------------------------------------------
# FINAL SOFT TEMPERATURE
# ------------------------------------------------

gamma = 0.965

p = p ** gamma
p = normalize_probs(p)

# ------------------------------------------------
# SAVE
# ------------------------------------------------

sub[cols] = p

SAVE_NAME = "submission_final_076.csv"

sub.to_csv(SAVE_NAME, index=False)

print("Saved:", SAVE_NAME)

print("\nAverage probabilities:")
print(sub[cols].mean())

print("\nMax probability:")
print(sub[cols].max().max())

print("\nRow sums:")
print(
    sub[cols].sum(axis=1).min(),
    sub[cols].sum(axis=1).max()
)

Saved: submission_final_076.csv

Average probabilities:
A_small    0.208251
A_big      0.240968
B_small    0.181035
B_big      0.369746
dtype: float64

Max probability:
0.40929367264500777

Row sums:
0.9999999999999998 1.0000000000000002


In [117]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings('ignore')

print("🚀 INITIALIZING END-TO-END GRANDMASTER PIPELINE...")

# ==========================================
# 1. LOAD RAW DATA & EXTRACT TARGETS
# ==========================================
print("📂 Loading train_IPL.csv and extracting target classes...")
df_raw = pd.read_csv('train_IPL.csv')

# Filter out abandoned matches
df_clean = df_raw[~df_raw['result_type'].isin(['tie', 'no result'])].copy()

# Extract final match states from ball-by-ball data
innings_final = df_clean.groupby(['Match ID', 'Innings']).last().reset_index()
match_df = innings_final.pivot(index='Match ID', columns='Innings',
                               values=['Innings Runs', 'Innings Wickets', 'Bat First', 'Bat Second', 'match_won_by', 'Date', 'Venue', 'toss_winner'])
match_df.columns = [f'{col[0]}_inn{col[1]}' for col in match_df.columns]
match_df = match_df.reset_index()

match_df = match_df.rename(columns={
    'Bat First_inn1': 'team_A',
    'Bat Second_inn1': 'team_B',
    'match_won_by_inn1': 'winner',
    'Date_inn1': 'Date',
    'Venue_inn1': 'Venue',
    'toss_winner_inn1': 'toss_winner'
})

# Build target_class (0=A_small, 1=A_big, 2=B_small, 3=B_big)
def compute_target(row):
    runs_1 = row['Innings Runs_inn1']
    runs_2 = row['Innings Runs_inn2']
    wks_2 = row['Innings Wickets_inn2']
    if pd.isna(runs_1) or pd.isna(runs_2): return np.nan
    if row['winner'] == row['team_A']:
        return 1 if (runs_1 - runs_2) > 20 else 0
    else:
        return 3 if (10 - wks_2) >= 6 else 2

match_df['target_class'] = match_df.apply(compute_target, axis=1)
train_matches = match_df.dropna(subset=['target_class']).copy()
train_matches['target_class'] = train_matches['target_class'].astype(int)
train_matches['Date'] = pd.to_datetime(train_matches['Date'])
train_matches = train_matches.sort_values('Date').reset_index(drop=True)

# ==========================================
# 2. ENGINEER GRANDMASTER FEATURES
# ==========================================
print("⚡ Engineering Tactical Features (H2H, Toss Bias, EMAs)...")

# Feature 1: Head-to-Head (H2H) Dominance
train_matches['matchup_id'] = train_matches.apply(lambda r: "-".join(sorted([str(r['team_A']), str(r['team_B'])])), axis=1)
def get_h2h(row, df):
    past_matches = df[(df['matchup_id'] == row['matchup_id']) & (df['Date'] < row['Date'])]
    if len(past_matches) == 0: return 0.5
    a_wins = past_matches[past_matches['winner'] == row['team_A']].shape[0]
    return a_wins / len(past_matches)

train_matches['H2H_TeamA_WinRate'] = train_matches.apply(lambda r: get_h2h(r, train_matches), axis=1)

# Feature 2: Toss Vulnerability Bias
train_matches['lost_toss'] = (train_matches['toss_winner'] != train_matches['winner']).astype(int)
train_matches['team_A_toss_resilience'] = train_matches.groupby('team_A')['lost_toss'].apply(lambda x: x.shift(1).expanding().mean()).reset_index(level=0, drop=True).fillna(0.5)
train_matches['team_B_toss_resilience'] = train_matches.groupby('team_B')['lost_toss'].apply(lambda x: x.shift(1).expanding().mean()).reset_index(level=0, drop=True).fillna(0.5)
train_matches['delta_toss_resilience'] = train_matches['team_A_toss_resilience'] - train_matches['team_B_toss_resilience']

# Feature 3: Venue Physics
train_matches['venue_par_score'] = train_matches.groupby('Venue')['Innings Runs_inn1'].apply(lambda x: x.shift(1).expanding().mean()).reset_index(level=0, drop=True).fillna(160)
train_matches['chase_won'] = (train_matches['winner'] == train_matches['team_B']).astype(int)
train_matches['venue_chase_bias'] = train_matches.groupby('Venue')['chase_won'].apply(lambda x: x.shift(1).expanding().mean()).reset_index(level=0, drop=True).fillna(0.5)

# Feature 4: Core Capability EMAs (10-match Rolling Windows)
train_matches['team_A_bat_runs_ema'] = train_matches.groupby('team_A')['Innings Runs_inn1'].transform(lambda x: x.shift(1).ewm(span=10).mean())
train_matches['team_B_bat_runs_ema'] = train_matches.groupby('team_B')['Innings Runs_inn2'].transform(lambda x: x.shift(1).ewm(span=10).mean())
train_matches['delta_bat_runs'] = train_matches['team_A_bat_runs_ema'] - train_matches['team_B_bat_runs_ema']

train_matches['team_A_bowl_wks_ema'] = train_matches.groupby('team_A')['Innings Wickets_inn2'].transform(lambda x: x.shift(1).ewm(span=10).mean())
train_matches['team_B_bowl_wks_ema'] = train_matches.groupby('team_B')['Innings Wickets_inn1'].transform(lambda x: x.shift(1).ewm(span=10).mean())
train_matches['delta_bowl_wks'] = train_matches['team_A_bowl_wks_ema'] - train_matches['team_B_bowl_wks_ema']

# Lock in the feature set
feature_cols = [
    'H2H_TeamA_WinRate', 'delta_toss_resilience',
    'venue_par_score', 'venue_chase_bias',
    'delta_bat_runs', 'delta_bowl_wks'
]

X_train = train_matches[feature_cols].fillna(train_matches[feature_cols].mean()).fillna(0)
y_train = train_matches['target_class']

# ==========================================
# 3. TRAIN LIGHTGBM ENSEMBLE
# ==========================================
print(f"🧠 Training Model on {X_train.shape[0]} Matches...")
lgb_params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.015,
    'num_leaves': 15,
    'max_depth': 4,
    'feature_fraction': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.5,
    'seed': 42,
    'verbose': -1
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
trained_models = []
oof_preds = np.zeros((len(X_train), 4))

for fold, (t_idx, v_idx) in enumerate(skf.split(X_train, y_train)):
    d_train = lgb.Dataset(X_train.iloc[t_idx], label=y_train.iloc[t_idx])
    d_val = lgb.Dataset(X_train.iloc[v_idx], label=y_train.iloc[v_idx], reference=d_train)

    clf = lgb.train(
        lgb_params, d_train, num_boost_round=1500,
        valid_sets=[d_train, d_val], callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_preds[v_idx] = clf.predict(X_train.iloc[v_idx], num_iteration=clf.best_iteration)
    trained_models.append(clf)

# Apply Algorithmic Epsilon Shield
oof_preds_clipped = np.clip(oof_preds, 0.015, 0.985)
oof_preds_clipped = oof_preds_clipped / np.sum(oof_preds_clipped, axis=1, keepdims=True)
print("="*50)
print(f"🏆 VALIDATED OUT-OF-FOLD LOG LOSS: {log_loss(y_train, oof_preds_clipped):.5f}")
print("="*50)

# ==========================================
# 4. PREDICT TEST MATCHES & EXPORT
# ==========================================
print("🔮 Predicting Future Outcomes...")
pub_df = pd.read_csv('public_lb_matches.csv').rename(columns={'match_id': 'Match ID', 'team_a': 'team_A', 'team_b': 'team_B', 'venue': 'Venue'})
sch_df = pd.read_csv('schedule.csv').rename(columns={'match_id': 'Match ID', 'team_a': 'team_A', 'team_b': 'team_B', 'venue': 'Venue'})
test_matches = pd.concat([pub_df, sch_df], ignore_index=True)

inf_data = []
for _, row in test_matches.iterrows():
    team_a, team_b = row['team_A'], row['team_B']
    row_feats = {}

    # H2H Inference
    matchup_id = "-".join(sorted([str(team_a), str(team_b)]))
    past_matches = train_matches[(train_matches['matchup_id'] == matchup_id)]
    row_feats['H2H_TeamA_WinRate'] = past_matches[past_matches['winner'] == team_a].shape[0] / len(past_matches) if len(past_matches) > 0 else 0.5

    # Latest Team Form Inference
    def get_latest_val(team, col_a, col_b):
        history = train_matches[(train_matches['team_A'] == team) | (train_matches['team_B'] == team)]
        if history.empty: return train_matches[col_a].mean()
        last = history.iloc[-1]
        return last[col_a] if last['team_A'] == team else last[col_b]

    toss_res_A = get_latest_val(team_a, 'team_A_toss_resilience', 'team_B_toss_resilience')
    toss_res_B = get_latest_val(team_b, 'team_A_toss_resilience', 'team_B_toss_resilience')
    row_feats['delta_toss_resilience'] = toss_res_A - toss_res_B

    bat_runs_A = get_latest_val(team_a, 'team_A_bat_runs_ema', 'team_B_bat_runs_ema')
    bat_runs_B = get_latest_val(team_b, 'team_A_bat_runs_ema', 'team_B_bat_runs_ema')
    row_feats['delta_bat_runs'] = bat_runs_A - bat_runs_B

    bowl_wks_A = get_latest_val(team_a, 'team_A_bowl_wks_ema', 'team_B_bowl_wks_ema')
    bowl_wks_B = get_latest_val(team_b, 'team_A_bowl_wks_ema', 'team_B_bowl_wks_ema')
    row_feats['delta_bowl_wks'] = bowl_wks_A - bowl_wks_B

    # Venue Inference
    v_stats = train_matches[train_matches['Venue'] == row['Venue']]
    row_feats['venue_par_score'] = v_stats['venue_par_score'].iloc[-1] if not v_stats.empty else train_matches['venue_par_score'].mean()
    row_feats['venue_chase_bias'] = v_stats['venue_chase_bias'].iloc[-1] if not v_stats.empty else 0.5

    inf_data.append(row_feats)

X_test = pd.DataFrame(inf_data)[feature_cols].fillna(X_train.mean())

# Generate final predictions
final_preds = np.zeros((len(X_test), 4))
for clf in trained_models:
    final_preds += clf.predict(X_test) / len(trained_models)

final_preds = np.clip(final_preds, 0.015, 0.985)
final_preds = final_preds / np.sum(final_preds, axis=1, keepdims=True)

out_df = pd.DataFrame({
    'match_id': test_matches['Match ID'],
    'A_small': final_preds[:, 0], 'A_big': final_preds[:, 1],
    'B_small': final_preds[:, 2], 'B_big': final_preds[:, 3]
})

sub_template = pd.read_csv('sample_submission.csv')
final_sub = sub_template[['match_id']].merge(out_df, on='match_id', how='left')
final_sub.to_csv('final_ml_submission.csv', index=False)

print("🎯 SUCCESS! 'final_ml_submission.csv' generated. Ready for Kaggle upload.")

🚀 INITIALIZING END-TO-END GRANDMASTER PIPELINE...
📂 Loading train_IPL.csv and extracting target classes...
⚡ Engineering Tactical Features (H2H, Toss Bias, EMAs)...
🧠 Training Model on 1124 Matches...
🏆 VALIDATED OUT-OF-FOLD LOG LOSS: 1.35477
🔮 Predicting Future Outcomes...
🎯 SUCCESS! 'final_ml_submission.csv' generated. Ready for Kaggle upload.


In [118]:
# ============================================================
# FINAL KAGGLE PUSH
# ============================================================

import pandas as pd
import numpy as np

def normalize_probs(p):
    p = np.clip(np.asarray(p), 1e-12, 1)
    return p / p.sum(axis=1, keepdims=True)

# LOAD BEST FILE
sub = pd.read_csv("submission_v8_soft_80.csv")

cols = ["A_small", "A_big", "B_small", "B_big"]

p = sub[cols].values.copy()

# ------------------------------------------------
# FINAL SOFT TEMPERATURE
# ------------------------------------------------

gamma = 0.965

p = p ** gamma
p = normalize_probs(p)

# ------------------------------------------------
# SAVE
# ------------------------------------------------

sub[cols] = p

SAVE_NAME = "submission_final_076.csv"

sub.to_csv(SAVE_NAME, index=False)

print("Saved:", SAVE_NAME)

print("\nAverage probabilities:")
print(sub[cols].mean())

print("\nMax probability:")
print(sub[cols].max().max())

print("\nRow sums:")
print(
    sub[cols].sum(axis=1).min(),
    sub[cols].sum(axis=1).max()
)

Saved: submission_final_076.csv

Average probabilities:
A_small    0.208251
A_big      0.240968
B_small    0.181035
B_big      0.369746
dtype: float64

Max probability:
0.40929367264500777

Row sums:
0.9999999999999998 1.0000000000000002


In [119]:
import os
import pandas as pd

FILE_NAME = "submission_final_076.csv"

# Check if file exists
if os.path.exists(FILE_NAME):
    print("FOUND:", FILE_NAME)

    # Read and verify
    sub = pd.read_csv(FILE_NAME)

    print("\nShape:", sub.shape)
    print("\nColumns:")
    print(sub.columns.tolist())

    print("\nFirst rows:")
    print(sub.head())

    # Validate probabilities
    prob_cols = ["A_small", "A_big", "B_small", "B_big"]

    print("\nRow sum min/max:")
    print(
        sub[prob_cols].sum(axis=1).min(),
        sub[prob_cols].sum(axis=1).max()
    )

    print("\nAny missing values:")
    print(sub.isnull().sum())

else:
    print("FILE NOT FOUND")

FOUND: submission_final_076.csv

Shape: (53, 5)

Columns:
['match_id', 'A_small', 'A_big', 'B_small', 'B_big']

First rows:
  match_id   A_small     A_big   B_small     B_big
0  1473488  0.219355  0.255389  0.168456  0.356800
1  1473489  0.206809  0.253979  0.167785  0.371426
2  1473490  0.210891  0.239285  0.161235  0.388588
3  1473491  0.207100  0.243423  0.169117  0.380360
4  1473492  0.220139  0.240688  0.167490  0.371683

Row sum min/max:
0.9999999999999996 1.0000000000000002

Any missing values:
match_id    0
A_small     0
A_big       0
B_small     0
B_big       0
dtype: int64


In [120]:
# ============================================================
# V12 FINAL CALIBRATED ENSEMBLE
# TARGET: < 1.33 PUBLIC LB
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# LOAD PREDICTIONS
# ------------------------------------------------------------
# Replace filenames if needed

v5 = pd.read_csv("submission_v5.csv")
v6 = pd.read_csv("submission_v6.csv")
v8 = pd.read_csv("submission_v8_soft_80.csv")

classes = ["A_small", "A_big", "B_small", "B_big"]

# ------------------------------------------------------------
# EXTRACT PROBABILITIES
# ------------------------------------------------------------

p_v5 = v5[classes].values
p_v6 = v6[classes].values
p_v8 = v8[classes].values

# ------------------------------------------------------------
# STEP 1: CONTROLLED ENSEMBLE
# ------------------------------------------------------------

p = (
    0.60 * p_v6 +
    0.25 * p_v5 +
    0.15 * p_v8
)

# ------------------------------------------------------------
# STEP 2: TEMPERATURE SOFTENING
# ------------------------------------------------------------

TEMP = 1.12

p = p ** (1 / TEMP)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# STEP 3: SAFE PROBABILITY CLIPPING
# ------------------------------------------------------------

MIN_PROB = 0.14
MAX_PROB = 0.39

p = np.clip(p, MIN_PROB, MAX_PROB)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# STEP 4: PRIOR CALIBRATION
# ------------------------------------------------------------

prior = np.array([
    0.21,  # A_small
    0.24,  # A_big
    0.19,  # B_small
    0.36   # B_big
])

BLEND = 0.08

p = (1 - BLEND) * p + BLEND * prior
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# CREATE FINAL SUBMISSION
# ------------------------------------------------------------

final_sub = v6.copy()

for i, c in enumerate(classes):
    final_sub[c] = p[:, i]

# ------------------------------------------------------------
# SAVE FILE
# ------------------------------------------------------------

FINAL_NAME = "submission_v12_final.csv"

final_sub.to_csv(FINAL_NAME, index=False)

# ------------------------------------------------------------
# VALIDATION CHECKS
# ------------------------------------------------------------

print("=" * 60)
print("AVERAGE CLASS PROBABILITIES")
print("=" * 60)

print(final_sub[classes].mean())

print("\nMax probability:")
print(final_sub[classes].max().max())

print("\nMin probability:")
print(final_sub[classes].min().min())

print("\nRow sums:")
print(
    final_sub[classes].sum(axis=1).min(),
    final_sub[classes].sum(axis=1).max()
)

print("\nDominant classes:")
print(final_sub[classes].idxmax(axis=1).value_counts())

print("\nSaved:", FINAL_NAME)

AVERAGE CLASS PROBABILITIES
A_small    0.178298
A_big      0.250281
B_small    0.193275
B_big      0.378146
dtype: float64

Max probability:
0.4288684504185487

Min probability:
0.14090395684919557

Row sums:
1.0 1.0

Dominant classes:
B_big    47
A_big     6
Name: count, dtype: int64

Saved: submission_v12_final.csv


In [121]:
# ============================================================
# V12.1 FINAL SAFE TOP-LB CALIBRATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# LOAD FILES
# ------------------------------------------------------------

v5 = pd.read_csv("submission_v5.csv")
v6 = pd.read_csv("submission_v6.csv")
v8 = pd.read_csv("submission_v8_soft_80.csv")

classes = ["A_small", "A_big", "B_small", "B_big"]

p5 = v5[classes].values
p6 = v6[classes].values
p8 = v8[classes].values

# ------------------------------------------------------------
# SOFTER ENSEMBLE
# ------------------------------------------------------------

p = (
    0.50 * p6 +
    0.30 * p5 +
    0.20 * p8
)

# ------------------------------------------------------------
# TEMPERATURE SOFTENING
# ------------------------------------------------------------

TEMP = 1.18

p = p ** (1 / TEMP)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# STRONGER SAFETY CLIP
# ------------------------------------------------------------

p = np.clip(p, 0.16, 0.37)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# PRIOR PULL
# ------------------------------------------------------------

prior = np.array([
    0.21,
    0.24,
    0.19,
    0.36
])

p = 0.88 * p + 0.12 * prior
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# BUILD SUBMISSION
# ------------------------------------------------------------

sub = v6.copy()

for i, c in enumerate(classes):
    sub[c] = p[:, i]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

NAME = "submission_v12_safe.csv"

sub.to_csv(NAME, index=False)

# ------------------------------------------------------------
# CHECKS
# ------------------------------------------------------------

print("=" * 60)
print("AVERAGE CLASS PROBABILITIES")
print("=" * 60)

print(sub[classes].mean())

print("\nMax probability:")
print(sub[classes].max().max())

print("\nDominant classes:")
print(sub[classes].idxmax(axis=1).value_counts())

print("\nRow sums:")
print(
    sub[classes].sum(axis=1).min(),
    sub[classes].sum(axis=1).max()
)

print("\nSaved:", NAME)

AVERAGE CLASS PROBABILITIES
A_small    0.189631
A_big      0.249567
B_small    0.195306
B_big      0.365497
dtype: float64

Max probability:
0.4021408193308419

Dominant classes:
B_big    51
A_big     2
Name: count, dtype: int64

Row sums:
1.0 1.0000000000000002

Saved: submission_v12_safe.csv


In [122]:
# ============================================================
# V13 RANK-DIVERSITY CALIBRATION
# ============================================================

import numpy as np
import pandas as pd

v5 = pd.read_csv("submission_v5.csv")
v6 = pd.read_csv("submission_v6.csv")
v8 = pd.read_csv("submission_v8_soft_80.csv")

classes = ["A_small", "A_big", "B_small", "B_big"]

p5 = v5[classes].values
p6 = v6[classes].values
p8 = v8[classes].values

# ------------------------------------------------------------
# BASE ENSEMBLE
# ------------------------------------------------------------

p = (
    0.45 * p6 +
    0.35 * p5 +
    0.20 * p8
)

# ------------------------------------------------------------
# TEMPERATURE
# ------------------------------------------------------------

TEMP = 1.22

p = p ** (1 / TEMP)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# PRIOR BLEND
# ------------------------------------------------------------

prior = np.array([
    0.21,
    0.24,
    0.19,
    0.36
])

p = 0.85 * p + 0.15 * prior
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# RANK DIVERSITY FIX
# ------------------------------------------------------------

# reduce B_big slightly
p[:, 3] *= 0.92

# boost A_big slightly
p[:, 1] *= 1.08

# boost B_small slightly
p[:, 2] *= 1.04

# normalize
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# SAFETY CLIP
# ------------------------------------------------------------

p = np.clip(p, 0.17, 0.36)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# BUILD SUBMISSION
# ------------------------------------------------------------

sub = v6.copy()

for i, c in enumerate(classes):
    sub[c] = p[:, i]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

NAME = "submission_v13_diverse.csv"

sub.to_csv(NAME, index=False)

# ------------------------------------------------------------
# CHECKS
# ------------------------------------------------------------

print("=" * 60)
print("AVERAGE CLASS PROBABILITIES")
print("=" * 60)

print(sub[classes].mean())

print("\nMax probability:")
print(sub[classes].max().max())

print("\nDominant classes:")
print(sub[classes].idxmax(axis=1).value_counts())

print("\nRow sums:")
print(
    sub[classes].sum(axis=1).min(),
    sub[classes].sum(axis=1).max()
)

print("\nSaved:", NAME)

AVERAGE CLASS PROBABILITIES
A_small    0.191137
A_big      0.266195
B_small    0.199658
B_big      0.343010
dtype: float64

Max probability:
0.3817374608512286

Dominant classes:
B_big    41
A_big    12
Name: count, dtype: int64

Row sums:
0.9999999999999999 1.0000000000000002

Saved: submission_v13_diverse.csv


In [123]:
# ============================================================
# V14 FINAL ELITE CALIBRATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# LOAD FILES
# ------------------------------------------------------------

v5 = pd.read_csv("submission_v5.csv")
v6 = pd.read_csv("submission_v6.csv")
v8 = pd.read_csv("submission_v8_soft_80.csv")

classes = ["A_small", "A_big", "B_small", "B_big"]

p5 = v5[classes].values
p6 = v6[classes].values
p8 = v8[classes].values

# ------------------------------------------------------------
# ENSEMBLE
# ------------------------------------------------------------

p = (
    0.42 * p6 +
    0.33 * p5 +
    0.25 * p8
)

# ------------------------------------------------------------
# SOFT TEMPERATURE
# ------------------------------------------------------------

TEMP = 1.24

p = p ** (1 / TEMP)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# PRIOR
# ------------------------------------------------------------

prior = np.array([
    0.21,
    0.24,
    0.19,
    0.36
])

p = 0.83 * p + 0.17 * prior
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# FINAL CLASS BALANCE
# ------------------------------------------------------------

# boost A_small
p[:, 0] *= 1.06

# slightly reduce A_big
p[:, 1] *= 0.97

# mild boost B_small
p[:, 2] *= 1.03

# slight reduce B_big
p[:, 3] *= 0.96

# normalize
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# SAFE CLIP
# ------------------------------------------------------------

p = np.clip(p, 0.18, 0.355)
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# BUILD SUBMISSION
# ------------------------------------------------------------

sub = v6.copy()

for i, c in enumerate(classes):
    sub[c] = p[:, i]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

NAME = "submission_v14_elite.csv"

sub.to_csv(NAME, index=False)

# ------------------------------------------------------------
# CHECKS
# ------------------------------------------------------------

print("=" * 60)
print("AVERAGE CLASS PROBABILITIES")
print("=" * 60)

print(sub[classes].mean())

print("\nMax probability:")
print(sub[classes].max().max())

print("\nDominant classes:")
print(sub[classes].idxmax(axis=1).value_counts())

print("\nRow sums:")
print(
    sub[classes].sum(axis=1).min(),
    sub[classes].sum(axis=1).max()
)

print("\nSaved:", NAME)

AVERAGE CLASS PROBABILITIES
A_small    0.206456
A_big      0.241766
B_small    0.202390
B_big      0.349388
dtype: float64

Max probability:
0.3809549708242781

Dominant classes:
B_big    53
Name: count, dtype: int64

Row sums:
0.9999999999999998 1.0000000000000002

Saved: submission_v14_elite.csv


In [124]:

# ============================================================
# V15 UNCERTAINTY DIVERSIFICATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

sub = pd.read_csv("submission_v14_elite.csv")

classes = ["A_small", "A_big", "B_small", "B_big"]

p = sub[classes].values.copy()

# ------------------------------------------------------------
# FIND UNCERTAIN ROWS
# ------------------------------------------------------------

# rows where top2 probs are close
sorted_p = np.sort(p, axis=1)

gap = sorted_p[:, -1] - sorted_p[:, -2]

# smaller gap = more uncertain
uncertain_idx = np.argsort(gap)

# ------------------------------------------------------------
# FORCE DIVERSITY
# ------------------------------------------------------------

# convert some rows to A_big dominance
for idx in uncertain_idx[:12]:

    boost = 0.035

    p[idx, 1] += boost   # A_big
    p[idx, 3] -= boost   # B_big

# convert some rows to B_small dominance
for idx in uncertain_idx[12:18]:

    boost = 0.025

    p[idx, 2] += boost   # B_small
    p[idx, 3] -= boost   # B_big

# ------------------------------------------------------------
# SAFETY CLIP
# ------------------------------------------------------------

p = np.clip(p, 0.18, 0.355)

# normalize
p = p / p.sum(axis=1, keepdims=True)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

for i, c in enumerate(classes):
    sub[c] = p[:, i]

NAME = "submission_v15_uncertainty.csv"

sub.to_csv(NAME, index=False)

# ------------------------------------------------------------
# CHECKS
# ------------------------------------------------------------

print("=" * 60)
print("AVERAGE CLASS PROBABILITIES")
print("=" * 60)

print(sub[classes].mean())

print("\nMax probability:")
print(sub[classes].max().max())

print("\nDominant classes:")
print(sub[classes].idxmax(axis=1).value_counts())

print("\nRow sums:")
print(
    sub[classes].sum(axis=1).min(),
    sub[classes].sum(axis=1).max()
)

print("\nSaved:", NAME)

AVERAGE CLASS PROBABILITIES
A_small    0.207892
A_big      0.251217
B_small    0.207439
B_big      0.333451
dtype: float64

Max probability:
0.3644595366401244

Dominant classes:
B_big    41
A_big    12
Name: count, dtype: int64

Row sums:
0.9999999999999999 1.0000000000000002

Saved: submission_v15_uncertainty.csv
